# SUV insights by price segment, brand and overall

This notebook answers the buyer questions behind the new web app. The app has three tabs, and this notebook has one part for each:

| App tab | What it shows | Notebook part |
|---|---|---|
| **Segments** | 26 questions for each price segment | Part 6 |
| **Brands** | The same 26 questions for each brand | Part 7 |
| **Overall** | 8 questions about all SUVs together | Part 8 |

Every answer comes with a **reason** that says why a car is ranked where it is, built from its own numbers. The answers are printed here, and also saved as CSV files in `insights_data/`, which the web app reads as they are.

**Inputs.** The notebook starts from three files that `suv_research.ipynb` wrote to `app_data/`:
- `variants_clean.csv`: one row per SUV variant, with clean prices, specs and yes/no features.
- `fair_prices.csv`: the fair price of each variant. The fair price comes from the price model in `suv_research.ipynb` (Step 4). It is what the market usually charges for that set of features, so a car priced below it is good value for money.
- `spec_sheet.csv`: CarDekho's own text for every feature, used for the few things the clean table doesn't hold.

Everything else (the segments, the scores, every ranking and every reason) is worked out here.

**How to read it.**
- **Parts 1 to 3** set up the data, the price segments, and the scores used to rank cars: value for money, features, modern features, engine performance and safety.
- **Part 4** lists the questions and the category each one belongs to.
- **Part 5** has one small function for each kind of question.
- **Parts 6 to 8** print the answers.
- **Part 9** checks them and saves the files.

## Part 1: Load the data

In [1]:
# Imports, file paths and settings for the whole notebook.
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

# The notebook sits in notebooks/, so the project folder is one level up.
PROJECT_DIR = Path.cwd().parent
APP_DATA_DIR = PROJECT_DIR / "app_data"      # inputs, written by suv_research.ipynb
OUT_DIR = PROJECT_DIR / "insights_data"       # outputs, read by the web app

for file_name in ["variants_clean.csv", "fair_prices.csv", "spec_sheet.csv"]:
    assert (APP_DATA_DIR / file_name).is_file(), f"Missing input: {APP_DATA_DIR / file_name}"
OUT_DIR.mkdir(exist_ok=True)

# Only an empty cell counts as missing. CarDekho uses the word "None" to mean
# "doesn't have it" (for example sunroof = None), and pandas would otherwise read
# that word as a missing value.
READ = dict(keep_default_na=False, na_values=[""])
KEY = ["company", "model", "variant"]
AS_TEXT = {column: str for column in KEY}   # variant names such as "200" must stay text

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

print("Inputs from: ", APP_DATA_DIR.name + "/")
print("Outputs to:  ", OUT_DIR.name + "/")
print("pandas", pd.__version__, "| numpy", np.__version__)

Inputs from:  app_data/
Outputs to:   insights_data/
pandas 3.1.1 | numpy 2.5.4


We join the variants to their fair prices on company, model and variant. Both tables must have exactly one row per variant, otherwise the join would silently duplicate or drop cars, so the cell checks that first.

In [2]:
variants_clean = pd.read_csv(APP_DATA_DIR / "variants_clean.csv", **READ, dtype=AS_TEXT)
fair_prices = pd.read_csv(APP_DATA_DIR / "fair_prices.csv", **READ, dtype=AS_TEXT)
spec_sheet = pd.read_csv(APP_DATA_DIR / "spec_sheet.csv", **READ, dtype=str)

print(f"variants_clean.csv: {len(variants_clean):,} variants, {variants_clean.shape[1]} columns")
print(f"fair_prices.csv:    {len(fair_prices):,} variants")
print(f"spec_sheet.csv:     {len(spec_sheet):,} rows (one per feature per variant)")

assert not variants_clean.duplicated(KEY).any(), "variants_clean has repeated variants"
assert not fair_prices.duplicated(KEY).any(), "fair_prices has repeated variants"

cars = variants_clean.merge(
    fair_prices[KEY + ["fair_price", "difference_pct", "badge", "missing_vs_price_peers"]],
    on=KEY, how="left", validate="one_to_one",
).reset_index(drop=True).copy()
assert len(cars) == len(variants_clean)

cars["name"] = cars["company"] + " " + cars["model"] + " " + cars["variant"]
cars["model_name"] = cars["company"] + " " + cars["model"]
assert cars["name"].is_unique

print(f"\nJoined table: {len(cars):,} variants of {cars['model_name'].nunique()} models "
      f"from {cars['company'].nunique()} brands")
print("Variants without an ex-showroom price:", cars["ex_showroom_price"].isna().sum())
print("Variants without a fair price:        ", cars["fair_price"].isna().sum())
print(cars.loc[cars["fair_price"].isna(), ["name", "ex_showroom_price", "badge"]].to_string(index=False))

variants_clean.csv: 1,399 variants, 232 columns
fair_prices.csv:    1,399 variants
spec_sheet.csv:     182,707 rows (one per feature per variant)

Joined table: 1,399 variants of 136 models from 25 brands
Variants without an ex-showroom price: 0
Variants without a fair price:         4
                                                name  ex_showroom_price           badge
         Land Rover Discovery 3.0 Diesel Dynamic HSE           12973000 Not enough data
Land Rover Discovery 3.0 Diesel Metropolitan Edition           13682000 Not enough data
                   Porsche Cayenne Electric Standard           17727000 Not enough data
                      Porsche Cayenne Electric Turbo           22751000 Not enough data


Every variant has an ex-showroom price. A few have no fair price, because the price model had too little data for them (their badge says "Not enough data"). They still appear in the segments and in the feature, safety and gearbox lists, but they are left out of every value-for-money ranking: without a fair price there is nothing to compare their price with.

## Part 2: Price segments

Buyers start with a budget, so the segments are **price bands in lakh**, the way Indian buyers and car magazines talk about them. We use the **ex-showroom price**, because it is listed for every variant and is the same in every city. On-road prices depend on the city and its taxes.

Each variant goes into a segment by its own price, so a model whose variants span several bands appears in each of them.

First, where do the prices sit?

In [3]:
price_lakh = cars["ex_showroom_price"] / 1e5
print("Ex-showroom price, in lakh:")
print(price_lakh.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2).to_string())

Ex-showroom price, in lakh:
count    1399.00
mean       32.16
std        54.50
min         5.69
10%         9.60
25%        12.61
50%        17.67
75%        24.00
90%        51.84
max       490.00


Half of all variants cost under about ₹17.7 lakh, and a long tail of luxury SUVs reaches several crore. A first try with the usual bands (under 10, 10–15, 15–20, 20–30, 30–50 and 50+ lakh):

In [4]:
first_try_edges = [0, 10, 15, 20, 30, 50, np.inf]
first_try = pd.cut(price_lakh, first_try_edges, right=False,
                   labels=["Under 10", "10-15", "15-20", "20-30", "30-50", "50+"])
first_try_table = (
    cars.assign(band=first_try, lakh=price_lakh)
    .groupby("band", observed=True)
    .agg(variants=("name", "size"), models=("model_name", "nunique"),
         cheapest_lakh=("lakh", "min"), dearest_lakh=("lakh", "max"))
    .round(2)
)
print(first_try_table.to_string())

top_band = cars[price_lakh >= 50].sort_values("ex_showroom_price")
print("\nThe 50+ band, cheapest and dearest models:")
print(top_band.groupby("model_name", sort=False)["ex_showroom_price"].min().div(1e5).round(1).iloc[[0, 1, 2, -3, -2, -1]].to_string())

          variants  models  cheapest_lakh  dearest_lakh
band                                                   
Under 10       185      20           5.69         10.00
10-15          346      45          10.09         15.00
15-20          363      39          15.04         20.00
20-30          283      27          20.06         29.95
30-50           79      22          30.04         50.00
50+            143      57          50.60        490.00

The 50+ band, cheapest and dearest models:
model_name
BYD Sealion 7                      50.6
Toyota Fortuner                    51.0
Mercedes-Benz GLA                  51.8
Mercedes-Benz Maybach GLS         275.0
Lexus LX                          281.4
Mercedes-Benz G-Class Electric    310.0


The first five bands work. The last one doesn't: it puts a ₹50 lakh SUV in the same group as cars costing several crore, and nobody shops across that range. So the top band is split at **₹1 crore**, which gives seven segments. Each one still holds enough variants and models to rank.

A variant belongs to a segment when its price is at least the lower bound and below the upper bound. So a car at exactly ₹10,00,000 is in "₹10–15 lakh".

In [5]:
SEGMENTS = pd.DataFrame(
    [
        (1, "Under ₹10 lakh", 0, 10),
        (2, "₹10–15 lakh", 10, 15),
        (3, "₹15–20 lakh", 15, 20),
        (4, "₹20–30 lakh", 20, 30),
        (5, "₹30–50 lakh", 30, 50),
        (6, "₹50 lakh–1 crore", 50, 100),
        (7, "Above ₹1 crore", 100, np.inf),
    ],
    columns=["segment_id", "segment", "from_lakh", "to_lakh"],
)
edges = list(SEGMENTS["from_lakh"]) + [np.inf]
cars["segment_id"] = pd.cut(price_lakh, edges, right=False, labels=SEGMENTS["segment_id"].tolist()).astype(int)
cars = cars.merge(SEGMENTS[["segment_id", "segment"]], on="segment_id", how="left", validate="many_to_one")
assert cars["segment"].notna().all()

def describe_groups(frame, by):
    """Variants, models, brands and the price range of each group."""
    cheapest = frame.loc[frame.groupby(by)["ex_showroom_price"].idxmin(), [by, "name"]].rename(columns={"name": "cheapest_variant"})
    table = (
        frame.groupby(by)
        .agg(variants=("name", "size"), models=("model_name", "nunique"), brands=("company", "nunique"),
             from_price=("ex_showroom_price", "min"), to_price=("ex_showroom_price", "max"))
        .reset_index()
        .merge(cheapest, on=by)
    )
    return table

segments = SEGMENTS.merge(describe_groups(cars, "segment_id"), on="segment_id")
print(segments.assign(from_lakh=segments["from_price"] / 1e5, to_lakh=segments["to_price"] / 1e5)
      [["segment", "variants", "models", "brands", "from_lakh", "to_lakh", "cheapest_variant"]].round(2).to_string(index=False))

         segment  variants  models  brands  from_lakh  to_lakh                         cheapest_variant
  Under ₹10 lakh       185      20      11       5.69    10.00                     Nissan Magnite Visia
     ₹10–15 lakh       346      45      13      10.09    15.00 Maruti Suzuki Brezza ZXi Turbo Dual Tone
     ₹15–20 lakh       363      39      13      15.04    20.00          Mahindra XUV 3XO AX7 L Turbo AT
     ₹20–30 lakh       283      27       9      20.06    29.95          Hyundai Creta King Turbo DCT DT
     ₹30–50 lakh        79      22      16      30.04    50.00 Tata Harrier EV Empowered QWD 75 Stealth
₹50 lakh–1 crore        51      31      15      50.60    99.85    BYD Sealion 7 Premium Festive Edition
  Above ₹1 crore        92      29       9     100.00   490.00   Mercedes-Benz GLE 300d 4Matic AMG Line


### Average price and spread of each segment

The price range shows where a segment starts and ends, but not where most of its cars sit. Two numbers sum that up, and the web app shows both at the top of each segment and on the home page:
- The **average price**: the mean ex-showroom price of the segment's variants.
- The **standard deviation**: how far a typical variant's price is from that average. A small one means the prices bunch together; a large one means they spread across the band.

Two choices behind these numbers:
1. **Each variant counts once.** A model with many variants in a segment pulls the average towards its prices more than a model with one. That fits the way the segments are built: a segment is the list of variants a buyer can choose from at that budget.
2. **Population standard deviation** (`ddof=0`). These are all the SUV variants CarDekho lists, not a sample of them, so we divide by the number of variants, not by one less. The table also shows the sample version (`ddof=1`, pandas' default), to check how much the choice matters.

The median is printed next to the average as a check on skew: when a few very expensive cars pull the average up, the median stays put.

In [6]:
price_stats = (
    cars.groupby("segment_id")["ex_showroom_price"]
    .agg(avg_price="mean", std_price=lambda p: p.std(ddof=0), sample_std=lambda p: p.std(ddof=1), median_price="median")
    .reset_index()
)
# The average of a band's prices must lie inside the band.
check = SEGMENTS.merge(price_stats, on="segment_id").merge(segments[["segment_id", "variants", "from_price", "to_price"]], on="segment_id")
assert check["avg_price"].between(check["from_price"], check["to_price"]).all()

segments = segments.merge(price_stats[["segment_id", "avg_price", "std_price"]], on="segment_id", validate="one_to_one")

stats_table = check.assign(
    average_lakh=check["avg_price"] / 1e5, std_lakh=check["std_price"] / 1e5,
    sample_std_lakh=check["sample_std"] / 1e5, median_lakh=check["median_price"] / 1e5,
    std_gap_pct=(check["sample_std"] / check["std_price"] - 1) * 100,
)
print(stats_table[["segment", "variants", "average_lakh", "std_lakh", "sample_std_lakh", "std_gap_pct", "median_lakh"]]
      .round(2).to_string(index=False))

         segment  variants  average_lakh  std_lakh  sample_std_lakh  std_gap_pct  median_lakh
  Under ₹10 lakh       185          8.71      1.06             1.06         0.27         8.92
     ₹10–15 lakh       346         12.66      1.37             1.37         0.14        12.72
     ₹15–20 lakh       363         17.70      1.50             1.50         0.14        17.87
     ₹20–30 lakh       283         23.69      2.39             2.39         0.18        23.66
     ₹30–50 lakh        79         38.15      5.54             5.57         0.64        37.69
₹50 lakh–1 crore        51         70.46     14.51            14.66         1.00        67.59
  Above ₹1 crore        92        209.46     95.40            95.92         0.55       171.64


What the numbers say:
- **The sample and population versions barely differ**: the biggest gap is 1.00%, in the smallest segment (₹50 lakh–1 crore, 51 variants). So the choice of `ddof` doesn't change any conclusion.
- **The five bands under ₹50 lakh are tight.** Their standard deviations run from ₹1.06 lakh to ₹5.54 lakh, small next to the width of each band, so most of their variants sit near the average.
- **Under ₹10 lakh averages ₹8.71 lakh**, near the top of its ₹5.69–10.00 lakh range: there are more variants near ₹10 lakh than near the cheapest cars.
- **Above ₹1 crore is the one segment where the average misleads.** It averages ₹209.46 lakh (about ₹2.09 crore), but its median is ₹171.64 lakh: the dearest cars, up to ₹4.90 crore, pull the average up. Its standard deviation of ₹95.40 lakh is large because the band has no upper limit. The web app shows the same two numbers, and its method page warns that the top segment's average sits above a typical car's price.

How many models have variants in more than one segment? This is why the segments are built from variants, not models:

In [7]:
spread = cars.groupby("model_name")["segment_id"].nunique()
print(f"{(spread > 1).sum()} of {len(spread)} models have variants in more than one segment.\n")
widest = spread.sort_values(ascending=False).index[0]
print(f"For example, {widest}:")
print(cars[cars["model_name"] == widest].groupby("segment")["ex_showroom_price"]
      .agg(variants="size", from_lakh="min", to_lakh="max")
      .assign(from_lakh=lambda t: (t["from_lakh"] / 1e5).round(2), to_lakh=lambda t: (t["to_lakh"] / 1e5).round(2))
      .reindex([s for s in SEGMENTS["segment"] if s in set(cars.loc[cars["model_name"] == widest, "segment"])])
      .to_string())

61 of 136 models have variants in more than one segment.

For example, Kia Seltos:
             variants  from_lakh  to_lakh
segment                                  
₹10–15 lakh         8      11.25    14.72
₹15–20 lakh        24      15.04    20.00
₹20–30 lakh         6      20.32    21.82


The web app's home page lists the segments, and clicking one shows its models and variants. That list is saved as `segment_models.csv`: one row per model per segment, with how many of its variants fall in that segment and their price range.

In [8]:
segment_models = (
    cars.groupby(["segment_id", "segment", "company", "model", "model_name"])
    .agg(variants=("name", "size"), from_price=("ex_showroom_price", "min"), to_price=("ex_showroom_price", "max"))
    .reset_index()
    .sort_values(["segment_id", "from_price", "model_name"])
)
models_per_segment = segment_models.groupby("segment", sort=False)["model_name"].apply(list)
for segment_name, model_list in models_per_segment.items():
    print(f"{segment_name} ({len(model_list)} models): " + textwrap.shorten(", ".join(model_list), 300, placeholder=" …"))

Under ₹10 lakh (20 models): Nissan Magnite, Tata Punch, Hyundai Exter, Renault Kiger, Maruti Suzuki FRONX, Tata Nexon, Toyota Taisor, Kia Sonet, Maruti Suzuki Brezza, Skoda Kylaq, Mahindra XUV 3XO, Hyundai Venue, Kia Syros, Citroen Basalt X, Mahindra Bolero, Citroen Aircross X, Mahindra Bolero Neo, Tata Punch EV, MG Astor, …
₹10–15 lakh (45 models): Maruti Suzuki Brezza, Tata Nexon, Hyundai Venue, Kia Sonet, Mahindra Thar OG, Mahindra Bolero Neo, Tata Punch EV, Renault Kiger, Toyota Taisor, Nissan Tekton, Renault Duster, Skoda Kylaq, Citroen Aircross X, Maruti Suzuki Victoris, Mahindra XUV 3XO, Maruti Suzuki FRONX, Kia Syros, Hyundai Venue N …
₹15–20 lakh (39 models): Mahindra XUV 3XO, Hyundai Creta, Kia Seltos, Maruti Suzuki Victoris, Toyota Urban Cruiser Hyryder, Tata Sierra, Maruti Suzuki Grand Vitara, Mahindra Thar ROXX, Renault Duster, Tata Curvv, MG Astor, Tata Nexon EV, Mahindra XUV 7XO, Honda Elevate, Nissan Tekton, Hyundai Venue N Line, Mahindra XUV400 …
₹20–30 lakh (27 models

## Part 3: The scores used to rank cars

The questions rank cars on five things: value for money, features, modern (eye-catching) features, engine performance and safety. Each is defined once here and used the same way everywhere after.

CarDekho doesn't list every feature for every car, so each feature is **yes (1)**, **no (0)** or **unknown (empty)**. The same rule applies throughout:
- A feature counts only when CarDekho confirms it. Unknown features earn no points.
- A basic feature counts as missing only when CarDekho says "no". An unknown never costs points.

So no car is punished for a gap in CarDekho's data, and none gets credit for a feature nobody confirmed.

### 3.1 Value for money

A variant's **difference** is how far its price is from its fair price: (price − fair price) ÷ fair price. A difference of −12% means it costs 12% less than the market usually charges for its features, which is good value. Positive means overpriced. The fair-price model's badge uses ±15%: more than 15% below is "Good value", more than 15% above is "Overpriced", and anything in between is "Fair".

In [9]:
priced = cars["fair_price"].notna()
print(f"Variants with a fair price: {priced.sum():,} of {len(cars):,}")
print("\nDifference from fair price (−0.12 = 12% below):")
print(cars.loc[priced, "difference_pct"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3).to_string())
print("\nBadges:", cars["badge"].value_counts().to_dict())

Variants with a fair price: 1,395 of 1,399

Difference from fair price (−0.12 = 12% below):
count    1395.000
mean        0.022
std         0.179
min        -0.414
5%         -0.176
25%        -0.075
50%        -0.001
75%         0.090
95%         0.249
max         1.463

Badges: {'Fair': 1091, 'Overpriced': 191, 'Good value': 113, 'Not enough data': 4}


### 3.2 Features: basics and the feature list

"Feature loaded" is measured with a **fixed list of features**, chosen so that CarDekho reports them (yes or no) for most variants and a buyer cares about them. Safety equipment such as airbags and ADAS is left out here: it has its own safety score in 3.5, so a car's safety kit isn't counted twice.

There are two lists:
- **Basic features** that any SUV should have: power steering, air conditioning, power windows and so on. A variant that lacks one loses **2 points**, because a missing basic matters more to a buyer than a missing extra.
- **The feature list**, which covers comfort, convenience, technology and convenience lighting. Each confirmed feature earns **1 point**.

**Feature score** = features on the list that the car has − 2 × basics it lacks.

A few features aren't in the clean table, so they are read from CarDekho's own text:
- **Power windows** come from the "Power Windows" row.
- **Ambient lighting** is any feature text that mentions ambient lighting.
- **Wireless Android Auto or Apple CarPlay** is the clean flag, or feature text that mentions it.
- **Premium sound** is 8 or more speakers, or a named sound brand such as Harman Kardon, Bose or JBL.

In [10]:
cars_key = cars[KEY]
spec_details = spec_sheet["feature_details"].fillna("").str.strip()
spec_names = spec_sheet["feature_name"].fillna("")

def flag(column):
    """A has_ column as 1 / 0 / NaN (NaN = CarDekho doesn't say)."""
    return cars[column].astype(float)

def from_values(column, yes, no):
    """1 if the column's value is in `yes`, 0 if in `no`, NaN otherwise."""
    out = pd.Series(np.nan, index=cars.index)
    out[cars[column].isin(no)] = 0.0
    out[cars[column].isin(yes)] = 1.0
    return out

def at_least(column, threshold):
    """1 if the number is at least `threshold`, 0 if below it, NaN if unknown."""
    values = cars[column].astype(float)
    return pd.Series(np.where(values.isna(), np.nan, (values >= threshold).astype(float)), index=cars.index)

def any_of(*flags):
    """1 if any flag is 1; 0 if none is 1 but one is 0; NaN if all are unknown."""
    frame = pd.concat(flags, axis=1)
    out = pd.Series(np.nan, index=cars.index)
    out[frame.eq(0).any(axis=1)] = 0.0
    out[frame.eq(1).any(axis=1)] = 1.0
    return out

def from_spec(row_mask, value):
    """`value` for every variant with at least one matching spec_sheet row, NaN for the rest."""
    hits = spec_sheet.loc[row_mask, KEY].drop_duplicates().assign(_value=value)
    return cars_key.merge(hits, on=KEY, how="left")["_value"].set_axis(cars.index)

power_window_rows = spec_names.eq("Power Windows")
power_windows = from_spec(power_window_rows & spec_details.isin(["Front & Rear", "Front Only"]), 1.0).fillna(
    from_spec(power_window_rows & spec_details.str.upper().eq("NO"), 0.0))

ambient_count = pd.to_numeric(spec_details.where(spec_names.eq("Ambient Light Colour (numbers)")), errors="coerce")
ambient_lighting = from_spec(
    (spec_details.str.contains("ambient", case=False) & ~spec_details.str.upper().eq("NO")) | ambient_count.gt(0), 1.0
).fillna(from_spec(spec_names.eq("Ambient Light Colour (numbers)") & (spec_details.str.upper().eq("NO") | ambient_count.eq(0)), 0.0))

wireless_projection = any_of(
    flag("has_wireless_phone_projection"),
    from_spec(spec_details.str.contains(r"wireless\s+(?:android|apple)", case=False, regex=True), 1.0),
)

SOUND_BRANDS = r"harman|bose|jbl|sony|infinity|meridian|bang\s*&?\s*olufsen|burmester|bowers|dynaudio|marantz|mark levinson|krell"
premium_sound = any_of(
    at_least("speakers", 8),
    from_spec(spec_details.str.contains(SOUND_BRANDS, case=False, regex=True), 1.0),
)

connected_car = any_of(flag("has_remote_vehicle_status_check"), flag("has_remote_ac_on_off"),
                       flag("has_over_the_air_ota_updates"), flag("has_in_car_remote_control_app"))
powered_tailgate = from_values("boot_opening", ["Powered", "Hands-free"], ["Manual", "Electronic"])

BASIC_FEATURES = {
    "power steering": flag("has_power_steering"),
    "air conditioning": flag("has_air_conditioner"),
    "power windows": power_windows,
    "central locking": flag("has_central_locking"),
    "remote keyless entry": flag("has_keyless_entry"),
    "Bluetooth": flag("has_bluetooth_connectivity"),
    "a touchscreen": flag("has_touchscreen"),
    "a height-adjustable driver seat": flag("has_height_adjustable_driver_seat"),
    "rear parking sensors": from_values("parking_sensors", ["Rear", "Front & Rear", "Yes, position not listed"], ["Front"]),
}

FEATURE_LIST = {
    "automatic climate control": flag("has_automatic_climate_control"),
    "rear AC vents": flag("has_rear_ac_vents"),
    "cruise control": flag("has_cruise_control"),
    "push-button start": flag("has_engine_start_stop_button"),
    "a rear camera": any_of(flag("has_rear_camera"), flag("has_360_view_camera")),
    "a 360° camera": flag("has_360_view_camera"),
    "front parking sensors": from_values("parking_sensors", ["Front & Rear", "Front"], ["Rear"]),
    "a digital instrument cluster": flag("has_digital_cluster"),
    "a touchscreen of 10 inches or more": at_least("touchscreen_in", 10),
    "6 or more speakers": at_least("speakers", 6),
    "wireless Android Auto / Apple CarPlay": wireless_projection,
    "wireless phone charging": flag("has_wireless_phone_charging"),
    "voice commands": flag("has_voice_commands"),
    "connected-car features": connected_car,
    "a sunroof": flag("has_sunroof"),
    "ventilated seats": flag("has_ventilated_seats"),
    "a powered driver seat": from_values("electric_seats", ["Driver only", "Front", "Front & Rear"], ["None"]),
    "leather or leatherette seats": from_values("upholstery", ["Leather", "Leatherette"], ["Fabric"]),
    "a leather-wrapped steering wheel": flag("has_leather_wrapped_steering_wheel"),
    "a rear centre armrest": flag("has_rear_seat_centre_arm_rest"),
    "a powered tailgate": powered_tailgate,
    "an air purifier": flag("has_air_quality_control"),
    "LED headlamps": flag("has_led_headlamps"),
    "automatic headlamps": flag("has_automatic_headlamps"),
    "rain-sensing wipers": flag("has_rain_sensing_wiper"),
    "ambient lighting": ambient_lighting,
    "alloy wheels": flag("has_alloy_wheels"),
    "a rear wiper": flag("has_rear_window_wiper"),
    "a rear defogger": flag("has_rear_window_defogger"),
}

def coverage(features):
    """Share of variants where CarDekho says yes, no, or nothing, for each feature."""
    frame = pd.DataFrame(features)
    return pd.DataFrame({"yes": frame.eq(1).mean(), "no": frame.eq(0).mean(), "unknown": frame.isna().mean()}).round(2)

print("Basic features (a confirmed 'no' costs 2 points):")
print(coverage(BASIC_FEATURES).to_string())
print(f"\nFeature list ({len(FEATURE_LIST)} features, 1 point each):")
print(coverage(FEATURE_LIST).to_string())

Basic features (a confirmed 'no' costs 2 points):
                                  yes    no  unknown
power steering                   1.00  0.00     0.00
air conditioning                 0.99  0.00     0.01
power windows                    0.84  0.00     0.16
central locking                  0.94  0.01     0.05
remote keyless entry             0.91  0.04     0.04
Bluetooth                        0.95  0.04     0.01
a touchscreen                    0.94  0.04     0.01
a height-adjustable driver seat  0.92  0.06     0.02
rear parking sensors             1.00  0.00     0.00

Feature list (29 features, 1 point each):
                                        yes    no  unknown
automatic climate control              0.83  0.14     0.03
rear AC vents                          0.92  0.04     0.03
cruise control                         0.79  0.17     0.04
push-button start                      0.82  0.12     0.05
a rear camera                          0.86  0.09     0.05
a 360° camera          

For the basics, CarDekho gives a clear yes or no for almost every variant, so a missing basic really is missing. Power windows are the exception: they are unknown for about one variant in six, and an unknown costs nothing. In the feature list, CarDekho reports most features for most cars. The ones it reports less often (ambient lighting, the air purifier, alloy wheels) still count only when confirmed.

In [11]:
basic_frame = pd.DataFrame(BASIC_FEATURES)
feature_frame = pd.DataFrame(FEATURE_LIST)
N_FEATURES = len(FEATURE_LIST)

cars["features_present"] = feature_frame.eq(1).sum(axis=1)
cars["basics_missing_count"] = basic_frame.eq(0).sum(axis=1)
cars["feature_score"] = cars["features_present"] - 2 * cars["basics_missing_count"]
cars["basics_missing"] = basic_frame.eq(0).apply(lambda row: "; ".join(row.index[row]), axis=1)
cars["feature_list"] = feature_frame.eq(1).apply(lambda row: "; ".join(row.index[row]), axis=1)

print(f"Feature score: features present (0 to {N_FEATURES}) minus 2 per missing basic")
print(cars["feature_score"].describe().round(1).to_string())
print(f"\nVariants missing at least one basic: {(cars['basics_missing_count'] > 0).sum()}")
print(cars.loc[cars["basics_missing_count"] > 0].sort_values("basics_missing_count", ascending=False)
      [["name", "basics_missing", "features_present", "feature_score"]].head(8).to_string(index=False))

Feature score: features present (0 to 29) minus 2 per missing basic
count    1399.0
mean       17.5
std         7.9
min        -8.0
25%        13.0
50%        19.0
75%        24.0
max        29.0

Variants missing at least one basic: 135
                     name                                                                                   basics_missing  features_present  feature_score
 Nissan Magnite Visia CNG central locking; remote keyless entry; Bluetooth; a touchscreen; a height-adjustable driver seat                 3             -7
       Mahindra Bolero B4                   power windows; central locking; remote keyless entry; Bluetooth; a touchscreen                 2             -8
 Nissan Magnite Visia AMT central locking; remote keyless entry; Bluetooth; a touchscreen; a height-adjustable driver seat                 3             -7
        Tata Safari Smart central locking; remote keyless entry; Bluetooth; a touchscreen; a height-adjustable driver seat                

### 3.3 Modern, eye-catching features

The question asks for features that are "very eye catching but not very important": things that impress in the showroom but don't change how safe or capable the car is. The list below picks those out: big screens, a panoramic sunroof, ambient lighting and similar comfort and gadget features. The **modern score** is how many of them a variant has.

In [12]:
MODERN_FEATURES = {
    "a touchscreen of 10.25 inches or more": at_least("touchscreen_in", 10.25),
    "a digital cluster of 10.25 inches or more": at_least("cluster_in", 10.25),
    "a panoramic sunroof": from_values("sunroof", ["Panoramic"], ["None", "Single"]),
    "ambient lighting": ambient_lighting,
    "ventilated seats": flag("has_ventilated_seats"),
    "wireless phone charging": flag("has_wireless_phone_charging"),
    "wireless Android Auto / Apple CarPlay": wireless_projection,
    "a 360° camera": flag("has_360_view_camera"),
    "a powered tailgate": powered_tailgate,
    "premium sound": premium_sound,
    "powered front seats": from_values("electric_seats", ["Front", "Front & Rear"], ["None", "Driver only"]),
    "a head-up display": flag("has_heads_up_display_hud"),
    "connected-car features": connected_car,
}
N_MODERN = len(MODERN_FEATURES)
modern_frame = pd.DataFrame(MODERN_FEATURES)
cars["modern_score"] = modern_frame.eq(1).sum(axis=1)
cars["modern_list"] = modern_frame.eq(1).apply(lambda row: "; ".join(row.index[row]), axis=1)

print(f"Modern features ({N_MODERN}):")
print(coverage(MODERN_FEATURES).to_string())
print("\nModern score by segment (median, out of", N_MODERN, "):")
print(cars.groupby("segment_id")["modern_score"].median().rename(index=dict(zip(SEGMENTS["segment_id"], SEGMENTS["segment"]))).to_string())

Modern features (13):
                                            yes    no  unknown
a touchscreen of 10.25 inches or more      0.55  0.41     0.04
a digital cluster of 10.25 inches or more  0.34  0.46     0.19
a panoramic sunroof                        0.51  0.31     0.18
ambient lighting                           0.39  0.03     0.58
ventilated seats                           0.44  0.37     0.19
wireless phone charging                    0.60  0.34     0.06
wireless Android Auto / Apple CarPlay      0.60  0.38     0.01
a 360° camera                              0.49  0.32     0.19
a powered tailgate                         0.24  0.51     0.26
premium sound                              0.34  0.62     0.04
powered front seats                        0.35  0.36     0.29
a head-up display                          0.10  0.22     0.69
connected-car features                     0.49  0.24     0.27

Modern score by segment (median, out of 13 ):
segment_id
Under ₹10 lakh       1.0
₹10–15 lakh  

### 3.4 Engine performance

Engine performance is measured by **power (bhp)** and **torque (Nm)**. Times for 0–100 km/h would be better, but CarDekho lists them for too few cars.

Power and torque use different units, so each is turned into a **percentile within the group being ranked** (a segment or a brand): 0 for the weakest variant there and 100 for the strongest. The **engine score** is the average of the two percentiles. "Strong" only means something next to the cars a buyer would compare with: 150 bhp is a lot under ₹15 lakh, but little above ₹1 crore. That's why the percentiles are taken within the group, not across all SUVs.

In the variant questions, "weak engine" means an engine score below 50 (weaker than at least half the group), and "rich in modern features" means a modern score above the group's median.

In [13]:
def percentile(values):
    """0 = lowest in the group, 100 = highest: the share of the other variants it beats (ties share the middle)."""
    known = values.notna().sum()
    if known <= 1:
        return pd.Series(np.where(values.notna(), 50.0, np.nan), index=values.index)
    return (values.rank(method="average") - 1) / (known - 1) * 100

def engine_score(group):
    """Average of the power and torque percentiles within the group."""
    return pd.concat([percentile(group["power_bhp"]), percentile(group["torque_nm"])], axis=1).mean(axis=1)

print("Power and torque known for:", (cars["power_bhp"].notna() & cars["torque_nm"].notna()).sum(), "variants;",
      "neither known for:", (cars["power_bhp"].isna() & cars["torque_nm"].isna()).sum())
example = cars[cars["segment_id"] == 2].assign(engine_score=lambda g: engine_score(g))
print("\nExample, ₹10–15 lakh: the strongest and weakest engines")
cols = ["name", "power_bhp", "torque_nm", "engine_score"]
print(pd.concat([example.nlargest(3, "engine_score"), example.nsmallest(3, "engine_score")])[cols].round(1).to_string(index=False))

Power and torque known for: 1381 variants; neither known for: 17

Example, ₹10–15 lakh: the strongest and weakest engines
                                  name  power_bhp  torque_nm  engine_score
                   Mahindra XUV 7XO AX      200.0      380.0          99.9
       Mahindra Scorpio N Z2 Petrol MT      200.0      370.0          99.8
                    Tata Harrier Smart      167.6      350.0          99.1
Tata Punch Accomplished Plus S CNG AMT       72.0      103.0           0.0
           Citroen Basalt X Basalt Max       80.0      115.0           1.4
     Hyundai Venue HX 5 Plus Dual Tone       82.0      114.7           1.6


### 3.5 Safety

Only about half the variants have a crash-test rating, so the safety score rests mainly on **safety equipment**, with the crash rating added as a bonus or penalty when one exists. The web app says the same.

**Equipment points (out of 100):**

| Equipment | Points |
|---|---|
| Airbags | 0 for 2 airbags, rising to 25 for 6, and 30 for 7 or more |
| ESC (electronic stability control) | 15 |
| ADAS (driver-assistance systems) | 15 |
| Tyre pressure monitor (TPMS) | 10 |
| Rear disc brakes | 10 |
| ISOFIX child-seat mounts | 5 |
| Hill-start assist | 5 |
| Camera: a 360° camera, or a rear camera | 10, or 5 |

**Crash rating:** Bharat NCAP is used when there is one, because it is India's own and current test. Otherwise Global NCAP is used.

| Rating | Adjustment |
|---|---|
| 5 stars | +10 |
| 4 stars | +5 |
| 3 stars | 0 |
| 2 stars or fewer | −10 |
| Not tested | 0 |

**Safety score** = equipment points + crash adjustment.

In [14]:
airbags = cars["airbags"]
airbag_points = (((airbags.clip(upper=6) - 2) / 4 * 25).clip(lower=0) + np.where(airbags >= 7, 5, 0)).fillna(0)

SAFETY_ITEMS = {  # label: (yes / no / unknown, points)
    "ESC": (flag("has_electronic_stability_control_esc"), 15),
    "ADAS": (flag("has_adas"), 15),
    "a tyre pressure monitor": (flag("has_tyre_pressure_monitoring_system_tpms"), 10),
    "rear disc brakes": (flag("has_rear_disc_brakes"), 10),
    "ISOFIX mounts": (flag("has_isofix_child_seat_mounts"), 5),
    "hill-start assist": (flag("has_hill_assist"), 5),
}
camera_360 = flag("has_360_view_camera").eq(1)
camera_rear = any_of(flag("has_rear_camera"), flag("has_360_view_camera")).eq(1)
camera_points = np.where(camera_360, 10, np.where(camera_rear, 5, 0))

equipment_points = airbag_points + camera_points
for label, (present, points) in SAFETY_ITEMS.items():
    equipment_points = equipment_points + present.eq(1) * points
cars["safety_equipment"] = equipment_points.round(1)

cars["crash_stars"] = cars["bharat_ncap_stars"].fillna(cars["global_ncap_stars"])
cars["crash_test"] = np.where(cars["bharat_ncap_stars"].notna(), "Bharat NCAP",
                              np.where(cars["global_ncap_stars"].notna(), "Global NCAP", ""))
def crash_adjustment(stars):
    if pd.isna(stars):
        return 0
    return 10 if stars >= 5 else 5 if stars >= 4 else 0 if stars >= 3 else -10
cars["crash_points"] = cars["crash_stars"].map(crash_adjustment)
cars["safety_score"] = cars["safety_equipment"] + cars["crash_points"]

safety_frame = pd.DataFrame({label: present for label, (present, _) in SAFETY_ITEMS.items()})
cars["safety_list"] = safety_frame.eq(1).apply(lambda row: "; ".join(row.index[row]), axis=1)
cars["safety_missing"] = safety_frame.ne(1).apply(lambda row: "; ".join(row.index[row]), axis=1)
cars["camera"] = np.where(camera_360, "a 360° camera", np.where(camera_rear, "a rear camera", ""))

print("Equipment coverage:")
print(coverage({label: present for label, (present, _) in SAFETY_ITEMS.items()}).to_string())
print("\nAirbags:", cars["airbags"].value_counts().sort_index().to_dict())
print("Crash ratings known:", (cars["crash_test"] != "").sum(), "variants;",
      cars["crash_test"].replace("", "not tested").value_counts().to_dict())
print("\nSafety score:")
print(cars["safety_score"].describe().round(1).to_string())

Equipment coverage:
                          yes    no  unknown
ESC                      0.96  0.01     0.03
ADAS                     0.42  0.36     0.22
a tyre pressure monitor  0.80  0.14     0.06
rear disc brakes         0.67  0.31     0.02
ISOFIX mounts            0.95  0.00     0.05
hill-start assist        0.86  0.01     0.13

Airbags: {2.0: 38, 6.0: 1111, 7.0: 157, 8.0: 27, 9.0: 26, 10.0: 27, 11.0: 6}
Crash ratings known: 765 variants; {'not tested': 634, 'Bharat NCAP': 615, 'Global NCAP': 150}

Safety score:
count    1399.0
mean       81.5
std        19.6
min       -10.0
25%        70.0
50%        85.0
75%        95.0
max       110.0


### 3.6 Gearbox, drive, seats, ADAS and engine size

- **Gearbox type** comes from the clean table: manual, AMT, CVT, DCT, torque converter, hybrid, single-speed (electric), or "automatic, type not listed" when CarDekho only says "automatic". Electric cars have no gearbox, so they are left out of the "where do automatics start" questions. Strong hybrids drive through their own hybrid gearbox, so they aren't counted as CVTs. Some unlisted types are filled in from the variant name, just below.
- **4x4 and AWD** are kept apart. The clean table merges them, so they are read from CarDekho's "Drive Type" row. "4WD" and "4x4" become **4x4**, and "AWD" stays **AWD**. These are the makers' own labels.
- **7-seater** means 7 or more seats (a few variants have 9).
- **ADAS** means CarDekho lists at least one driver-assistance feature (the clean `has_adas` flag). Cars without an ADAS section on CarDekho are treated as having no ADAS.
- **Engine size** is the engine's cc, grouped to the nearest 0.1 litre (1,197 cc becomes 1.2 litre). Electric cars form their own group.

In [15]:
GEARBOX_LABEL = {
    "Manual": "Manual",
    "AMT": "AMT",
    "CVT": "CVT",
    "DCT": "DCT",
    "Torque-converter AT": "Torque converter",
    "Hybrid": "Hybrid",
    "Automatic, type not listed": "Automatic (type not listed)",
    "Single-speed (EV)": "Single-speed (electric)",
}
assert set(cars["gearbox_type"].dropna()) <= set(GEARBOX_LABEL), set(cars["gearbox_type"]) - set(GEARBOX_LABEL)
cars["gearbox"] = cars["gearbox_type"].map(GEARBOX_LABEL)
cars["is_automatic"] = ~cars["gearbox_type"].isin(["Manual", "Single-speed (EV)"])
print("Gearbox types:", cars["gearbox"].value_counts().to_dict())

drive_rows = spec_sheet.loc[spec_names.eq("Drive Type"), KEY].assign(drive_raw=spec_details[spec_names.eq("Drive Type")].str.upper())
assert not drive_rows.duplicated(KEY).any()
DRIVE_LABEL = {"AWD": "AWD", "4WD": "4x4", "4X4": "4x4", "FWD": "2WD", "RWD": "2WD", "2WD": "2WD", "RWD(WITH MTT)": "2WD"}
assert set(drive_rows["drive_raw"]) <= set(DRIVE_LABEL), set(drive_rows["drive_raw"]) - set(DRIVE_LABEL)
cars = cars.merge(drive_rows, on=KEY, how="left", validate="one_to_one")
cars["drive_type"] = cars["drive_raw"].map(DRIVE_LABEL)
print("\nCarDekho's drive labels:", drive_rows["drive_raw"].value_counts().to_dict())
print("Drive type used here:   ", cars["drive_type"].value_counts(dropna=False).to_dict())
# The clean table's merged 4x4/AWD flag must agree with the split one.
both_known = cars["drive"].notna() & cars["drive_type"].notna()
assert (cars.loc[both_known, "drive"].eq("4x4/AWD") == cars.loc[both_known, "drive_type"].isin(["4x4", "AWD"])).all()

def fuel_label(row):
    if pd.isna(row["hybrid_type"]):
        return row["fuel_type"]
    return f"{row['fuel_type']} {'plug-in' if row['hybrid_type'] == 'Plug-in' else row['hybrid_type'].lower()} hybrid"
cars["fuel"] = cars.apply(fuel_label, axis=1)

cars["seven_plus"] = cars["seats"] >= 7
cars["adas"] = cars["has_adas"].eq(1)
cars["engine_size"] = np.where(cars["fuel_type"].eq("Electric"), "Electric",
                               (cars["engine_cc"] / 100).round().div(10).map(lambda x: f"{x:.1f} litre" if pd.notna(x) else None))
print("\nSeats:", cars["seats"].value_counts().sort_index().to_dict())
print("With ADAS:", cars["adas"].sum(), "variants")
print("Engine sizes:", cars["engine_size"].value_counts().sort_index().to_dict())
assert cars["engine_size"].notna().all()

Gearbox types: {'Manual': 553, 'Automatic (type not listed)': 338, 'Single-speed (electric)': 178, 'Torque converter': 129, 'DCT': 92, 'CVT': 48, 'AMT': 35, 'Hybrid': 26}

CarDekho's drive labels: {'FWD': 983, 'AWD': 158, 'RWD': 133, '2WD': 66, '4WD': 50, '4X4': 4, 'RWD(WITH MTT)': 1}
Drive type used here:    {'2WD': 1183, 'AWD': 158, '4x4': 54, nan: 4}

Seats: {4: 27, 5: 1076, 6: 66, 7: 227, 9: 3}
With ADAS: 585 variants
Engine sizes: {'1.0 litre': 141, '1.2 litre': 201, '1.3 litre': 21, '1.5 litre': 494, '1.6 litre': 16, '1.8 litre': 3, '1.9 litre': 2, '2.0 litre': 169, '2.2 litre': 79, '2.4 litre': 1, '2.5 litre': 5, '2.7 litre': 1, '2.8 litre': 10, '2.9 litre': 6, '3.0 litre': 34, '3.3 litre': 4, '4.0 litre': 8, '4.4 litre': 19, '5.0 litre': 7, 'Electric': 178}


**Filling in unlisted gearbox types.** The output above shows a problem for the gearbox questions: many automatics are "type not listed", so they would be missing from the DCT, CVT and torque-converter answers. But the variant name often says which kind it is: "Visia **AMT**", "Creative **DCA**" (Tata's name for its dual-clutch), "**IVT**" (Hyundai's name for its CVT). Two rules fill the gaps:

1. **The name says it.** AMT becomes AMT. DCT, DCA or DSG become DCT. CVT or IVT become CVT. First we check that this rule agrees with CarDekho wherever CarDekho does list the type. The only disagreements are strong hybrids named "… Hybrid CVT", which CarDekho files as hybrids because their hybrid gearbox only works like a CVT. So rule 1 is not used for strong hybrids.
2. **A plain "AT" in the name.** "AT" only says "automatic", so it gets the type of the same model's other "AT" variants, but only when all of those have one and the same type. Otherwise it stays "type not listed".

In [16]:
NAME_TYPE = {"AMT": "AMT", "DCT": "DCT", "DCA": "DCT", "DSG": "DCT", "CVT": "CVT", "IVT": "CVT"}
UNLISTED = "Automatic (type not listed)"
cars["gearbox_word"] = cars["variant"].str.upper().str.extract(r"\b(AMT|DCT|DCA|DSG|CVT|IVT|AT)\b", expand=False)

# Check rule 1 where CarDekho does name the type.
named = cars[cars["gearbox_word"].isin(NAME_TYPE) & cars["is_automatic"] & (cars["gearbox"] != UNLISTED)]
agree = (named["gearbox_word"].map(NAME_TYPE) == named["gearbox"])
print(f"Rule 1 check: the name and CarDekho agree for {agree.sum()} of {len(named)} automatics where both say the type.")
if (~agree).any():
    print(named.loc[~agree, ["name", "gearbox"]].to_string(index=False))

unlisted_before = (cars["gearbox"] == UNLISTED).sum()
# Strong hybrids are left out: their names may say "CVT", but they drive through a hybrid gearbox (see the check above).
by_name = (cars["gearbox"] == UNLISTED) & cars["gearbox_word"].isin(NAME_TYPE) & ~cars["hybrid_type"].eq("Strong")
cars.loc[by_name, "gearbox"] = cars.loc[by_name, "gearbox_word"].map(NAME_TYPE)

listed_at = cars[(cars["gearbox_word"] == "AT") & cars["is_automatic"] & (cars["gearbox"] != UNLISTED)]
at_type = listed_at.groupby("model_name")["gearbox"].agg(lambda s: s.iloc[0] if s.nunique() == 1 else None).dropna()
by_sibling = (cars["gearbox"] == UNLISTED) & cars["gearbox_word"].eq("AT") & cars["model_name"].isin(at_type.index)
cars.loc[by_sibling, "gearbox"] = cars.loc[by_sibling, "model_name"].map(at_type)
cars["gearbox_source"] = np.select([by_name, by_sibling], ["variant name", "same model's AT variants"], "CarDekho")

print(f"\nAutomatics with no type: {unlisted_before} before, {(cars['gearbox'] == UNLISTED).sum()} after")
print(f"  filled from the name (rule 1):              {by_name.sum()}  {cars.loc[by_name, 'gearbox'].value_counts().to_dict()}")
print(f"  filled from the model's AT variants (rule 2): {by_sibling.sum()}  {cars.loc[by_sibling, 'gearbox'].value_counts().to_dict()}")
print("\nExamples:")
print(cars.loc[by_name | by_sibling, ["name", "gearbox", "gearbox_source"]].drop_duplicates("gearbox").to_string(index=False))
still = cars[cars["gearbox"] == UNLISTED]
print(f"\nStill not listed, by model (top 8): {still['model_name'].value_counts().head(8).to_dict()}")
print("\nGearbox types now:", cars["gearbox"].value_counts().to_dict())

Rule 1 check: the name and CarDekho agree for 140 of 149 automatics where both say the type.
                                                   name gearbox
   Maruti Suzuki Grand Vitara Alpha Plus Opt Hybrid CVT  Hybrid
       Maruti Suzuki Grand Vitara Delta Plus Hybrid CVT  Hybrid
        Maruti Suzuki Grand Vitara Zeta Plus Hybrid CVT  Hybrid
     Maruti Suzuki Grand Vitara Zeta Plus Hybrid CVT DT  Hybrid
    Maruti Suzuki Grand Vitara Zeta Plus Opt Hybrid CVT  Hybrid
 Maruti Suzuki Grand Vitara Zeta Plus Opt Hybrid CVT DT  Hybrid
       Maruti Suzuki Grand Vitara Alpha Plus Hybrid CVT  Hybrid
    Maruti Suzuki Grand Vitara Alpha Plus Hybrid CVT DT  Hybrid
Maruti Suzuki Grand Vitara Alpha Plus Opt Hybrid CVT DT  Hybrid

Automatics with no type: 338 before, 267 after
  filled from the name (rule 1):              50  {'DCT': 29, 'AMT': 21}
  filled from the model's AT variants (rule 2): 21  {'Torque converter': 21}

Examples:
                       name          gearbox           gea

All the scores are saved per variant in `variants.csv`, so the web app can show any car's numbers next to its rank. The feature lists and safety points go to `feature_lists.csv`, so the app's About page can explain the scores.

In [17]:
VARIANT_COLUMNS = [
    "segment_id", "segment", "company", "model", "variant", "name", "model_name",
    "ex_showroom_price", "on_road_price", "fuel", "fuel_type", "gearbox", "gearbox_source", "is_automatic", "drive_type",
    "seats", "engine_cc", "engine_size", "battery_kwh", "power_bhp", "torque_nm", "airbags", "adas",
    "crash_stars", "crash_test", "fair_price", "difference_pct", "badge",
    "feature_score", "features_present", "basics_missing_count", "basics_missing", "feature_list",
    "modern_score", "modern_list", "safety_score", "safety_equipment", "crash_points", "safety_list",
    "cardekho_url", "price_date",
]
variants_out = cars[VARIANT_COLUMNS].sort_values(["segment_id", "ex_showroom_price", "name"]).reset_index(drop=True)

feature_lists = pd.concat([
    pd.DataFrame({"list": "basic", "feature": list(BASIC_FEATURES), "points": -2}),
    pd.DataFrame({"list": "feature", "feature": list(FEATURE_LIST), "points": 1}),
    pd.DataFrame({"list": "modern", "feature": list(MODERN_FEATURES), "points": 1}),
    pd.DataFrame({"list": "safety", "feature": ["airbags (2 → 0, 6 → 25, 7+ → 30)"] + list(SAFETY_ITEMS) + ["a 360° camera (or a rear camera: 5)"],
                  "points": [30] + [p for _, p in SAFETY_ITEMS.values()] + [10]}),
    pd.DataFrame({"list": "crash rating", "feature": ["5 stars", "4 stars", "3 stars", "2 stars or fewer", "not tested"],
                  "points": [10, 5, 0, -10, 0]}),
], ignore_index=True)
print(variants_out.shape, "variants.csv |", feature_lists.shape, "feature_lists.csv")
print(feature_lists.groupby("list", sort=False).size().to_string())

(1399, 41) variants.csv | (64, 3) feature_lists.csv
list
basic            9
feature         29
modern          13
safety           8
crash rating     5


## Part 4: The questions and their categories

The questions are the ones from the app brief. In the app they are grouped into **categories** (value for money, features, safety, gearbox type and so on), but the questions themselves stay as asked:
- `asked_as` keeps the original wording.
- `question` is the same question with only spelling and punctuation tidied, for display.

The **Brands** tab asks the segment questions again, with "segment" read as "brand".

Each question also has:
- **`kind`**: which answer function in Part 5 handles it.
- **`list_size`**: how many answers it asks for. When a segment or brand has fewer qualifying cars than the list asks for, only the single best one is shown. This follows the brief's rule for small groups, so a "top 10" never means "all 2 cars this brand makes".

The brief asks "which model has the most high safety feature variants" twice. Both are kept, as asked, and they share one answer.

In [18]:
SEGMENT_QUESTIONS = [
    # number, category, kind, list size, display text, as asked
    (1, "Value for money", "value_top", 10,
     "10 most value for money variants, in order",
     "10 Most value for money variant in order"),
    (2, "Features", "features_top", 10,
     "10 most feature loaded variants, ranked with reasons. Among the 3 most feature loaded, which is the most value for money?",
     "10 most feature loaded variants. Also among these 3 most feature loaded variants which is the most value for money. Rank these 10 with reason"),
    (3, "Value for money", "value_bottom", 10,
     "Which 10 variants provide the least value for money, and why?",
     "Which 10 variants provide the least value for money and why with reason."),
    (4, "Safety", "safety_top", 10,
     "10 best in safety, with reasons",
     "10 best in safety with reason"),
    (5, "Safety", "safety_bottom", 10,
     "10 worst in safety, with reasons",
     "10 worst in safety wit h reason"),
    (6, "4x4 and AWD", "drive", 0,
     "Which offer 4x4 and which offer AWD (all-wheel drive)?",
     "Which offers 4x4 and which offers AWD (all wheel drive)."),
    (7, "Modern features vs engine", "modern_weak_engine", 10,
     "Which 10 variants have the modern features (very eye-catching but not very important) but are weak in engine performance?",
     "Which 10 variants have the modern features (which are very eye catching but not very important) but are weak at engine features"),
    (8, "Modern features vs engine", "engine_weak_modern", 10,
     "Which 10 variants have strong engine performance but are weak in modern features?",
     "Which 10 variants have strong engine performance but weak at modern features."),
    (9, "Value for money", "model_value_top", 2,
     "Which are the 2 most value for money models in the segment? (models, not variants)",
     "Inside each segment which is the 2 most value for money car in general. Above we have discussed at a granular level based on different variants of different models, but here we will say which are the 2 most value for money models in the segment."),
    (10, "Value for money", "model_value_bottom", 1,
     "Which is the least value for money model in the segment? (models, not variants)",
     "Inside each segment which is the least value for money car in general. Above we have discussed at a granular level based on different variants of different models, but here we will say which are the least value for money models in the segment."),
    (11, "Features", "model_most_loaded", 1,
     "Which model has the most feature loaded variants in the segment? (model level)",
     "Which model (remember I am saying model here not variant , so in this case it should be a model level comparison) has in general the most number of  feature loaded variants in a segment."),
    (12, "Modern features vs engine", "model_engine_top", 2,
     "Which 2 models have the best engine performance in the segment?",
     "Which 2 models have the best engine performances in general in a segment."),
    (13, "Modern features vs engine", "model_modern_weak_engine", 2,
     "Which 2 models are rich in modern features but weak in engine performance? (models, not variants)",
     "Which 2 two models are modern feature rich but weak at engine performance. (not variant but model overall)"),
    (14, "Features", "model_features_top", 2,
     "Which 2 models have the most features (the most feature-rich variants)?",
     "Which 2 models in general are having the most number of features (meaning which models are having the most feature rich variants)"),
    (15, "Top and bottom of the price range", "pricey_picks", 2,
     "If we want to buy one of the most expensive variants in the segment, which 2 variants should we go for, and why?",
     "If we want to buy one of the most expensive variants (here i am talking about variant levels not a model in general) in the segment then which 2 variants should we go for and why ?"),
    (16, "Top and bottom of the price range", "cheap_picks", 2,
     "If we want to buy one of the least expensive variants in the segment, which 2 variants should we go for, and why?",
     "If we want to buy one of the least expensive variants (here i am talking about variant levels not a model in general) in the segment then which 2 variants should we go for and why ?"),
    (17, "Safety", "model_most_safe", 1,
     "Which model has the most high-safety variants? (model level)",
     "Earlier we compared at variant levels, but now tell me which model has the most number of high safety feature variants."),
    (18, "Safety", "model_most_safe", 1,
     "Which model has the most high-safety variants? (model level)",
     "Earlier we compared at variant levels, but now tell me which model has the most number of high safety feature variants."),
    (19, "Gearbox type", "auto_start", 10,
     "At what price does the automatic gearbox start? First 10 variants, with the type of gearbox",
     "At what price automatic gear box starts. List first 10 variants. Along with type of gearbox."),
    (20, "Gearbox type", "gearbox_start:DCT", 3,
     "At what price does the DCT automatic gearbox start? First 3 variants in the segment",
     "At what price DCT automatic gear box starts. List first 3 variants In a segment."),
    (21, "Gearbox type", "gearbox_start:CVT", 3,
     "At what price does the CVT automatic gearbox start? First 3 variants in the segment",
     "At what price CVT automatic gear box starts. List first 3 variants in a segment "),
    (22, "Gearbox type", "gearbox_start:Torque converter", 3,
     "At what price does the torque converter gearbox start? 3 variants in the segment",
     "At what price Torque converter gear box starts. List 3 variants in a segment."),
    (23, "Seats", "seven_seaters", 0,
     "Which cars have 7-seater variants in the segment? Are all variants of that model 7-seaters, or only some?",
     "List cars which has 7 seater variants in a segment. Also mention whether all the cars are 7 seaters for that model or some."),
    (24, "ADAS", "adas_cheapest", 5,
     "5 most affordable variants in the segment with ADAS",
     "5 Most affordable variants in asegment with ADAS"),
    (25, "ADAS", "adas_value", 2,
     "2 most value for money variants with ADAS in the segment",
     "2 most value for money varints with ADAS in a segment."),
    (26, "Engine size", "engine_sizes", 0,
     "Range of engine capacity (like 1500 cc) in the segment, with starting price",
     "In each segment print the range of engine capacity (like 1500cc ) along with starting price."),
]
OVERALL_QUESTIONS = [
    (1, "Gearbox type", "gearbox_start:DCT", 3,
     "At what price does the DCT automatic gearbox start? First 3 variants overall",
     "At what price DCT automatic gear box starts. List first 3 variants In a overall"),
    (2, "Gearbox type", "gearbox_start:CVT", 3,
     "At what price does the CVT automatic gearbox start? First 3 variants overall",
     "At what price CVT automatic gear box starts. List first 3 variants in a overall"),
    (3, "Gearbox type", "gearbox_start:Torque converter", 3,
     "At what price does the torque converter gearbox start? 3 variants overall",
     "At what price Torque converter gear box starts. List 3 variants in a overall."),
    (4, "ADAS", "models_adas_cheapest", 10,
     "10 most affordable models overall (not by segment) with ADAS",
     "10 most affordable models overall (not in a segment ) with adas."),
    (5, "Value for money by brand", "brand_value:5", 3,
     "Which brand has the most value for money 5-seater cars, and why?",
     "Overall which brand has the most value for money 5 seater cars along with reason."),
    (6, "Value for money by brand", "brand_value:7", 3,
     "Which brand has the most value for money 7-seater cars, and why?",
     "Overall which brand has the most value for money 7 seater cars along with reason."),
    (7, "Engine size", "engine_sizes", 0,
     "Range of engine capacity (like 1500 cc), with starting price",
     "Range of engine capacity (like 1500cc ) along with starting price."),
    (8, "4x4 and AWD", "drive_start", 10,
     "At what price does 4x4 or AWD (all-wheel drive) start? 10 most affordable variants",
     "At what price does does 4x4 or AWD (all wheel) drive start. List 10 most affordable variants."),
]
SEGMENT_CATEGORIES = ["Value for money", "Features", "Modern features vs engine", "Safety", "ADAS",
                      "Top and bottom of the price range", "Gearbox type", "Seats", "4x4 and AWD", "Engine size"]
OVERALL_CATEGORIES = ["Gearbox type", "ADAS", "Value for money by brand", "Engine size", "4x4 and AWD"]

def question_table(tab, prefix, questions, categories, replace=None):
    frame = pd.DataFrame(questions, columns=["number", "category", "kind", "list_size", "question", "asked_as"])
    if replace:
        frame["question"] = frame["question"].str.replace(*replace, regex=False)
    frame.insert(0, "question_id", [f"{prefix}{n:02d}" for n in frame["number"]])
    frame.insert(0, "tab", tab)
    frame["category_order"] = frame["category"].map({c: i for i, c in enumerate(categories, 1)})
    assert frame["category_order"].notna().all()
    return frame.sort_values(["category_order", "number"]).reset_index(drop=True)

questions = pd.concat([
    question_table("segment", "S", SEGMENT_QUESTIONS, SEGMENT_CATEGORIES),
    question_table("brand", "B", SEGMENT_QUESTIONS, SEGMENT_CATEGORIES, replace=("segment", "brand")),
    question_table("overall", "O", OVERALL_QUESTIONS, OVERALL_CATEGORIES),
], ignore_index=True)
questions["order"] = questions.groupby("tab").cumcount() + 1

for tab in ["segment", "overall"]:
    print(f"\n{tab.upper()} TAB")
    for category, block in questions[questions["tab"] == tab].groupby("category", sort=False):
        print(f"  {category}")
        for q in block.itertuples():
            print(f"    {q.question_id}  {q.question}")
print("\nBRAND TAB: the same", (questions["tab"] == "brand").sum(), "questions as the segment tab, with 'segment' read as 'brand'.")


SEGMENT TAB
  Value for money
    S01  10 most value for money variants, in order
    S03  Which 10 variants provide the least value for money, and why?
    S09  Which are the 2 most value for money models in the segment? (models, not variants)
    S10  Which is the least value for money model in the segment? (models, not variants)
  Features
    S02  10 most feature loaded variants, ranked with reasons. Among the 3 most feature loaded, which is the most value for money?
    S11  Which model has the most feature loaded variants in the segment? (model level)
    S14  Which 2 models have the most features (the most feature-rich variants)?
  Modern features vs engine
    S07  Which 10 variants have the modern features (very eye-catching but not very important) but are weak in engine performance?
    S08  Which 10 variants have strong engine performance but are weak in modern features?
    S12  Which 2 models have the best engine performance in the segment?
    S13  Which 2 models are ric

## Part 5: How each question is answered

Each kind of question has one small function. A function takes a **group** (one segment, one brand, or all SUVs) and returns its answer rows. Every row says what it ranks (a variant, model, brand or engine size), its rank, the price, one key number (`metric`), and a **reason** in plain words.

Some questions also get a **note** row (rank 0) that sums up the answer or explains why a list is short. For example, "DCT automatics start at ₹9.99 lakh here" or "No 4x4 variants in this segment".

How each kind is decided:

| Kind | Ranked by |
|---|---|
| Most / least value for money (variants) | Difference from fair price, lowest / highest first |
| Most feature loaded | Feature score, then better value. Of the top 3, the best value is marked |
| Best / worst safety | Safety score, highest / lowest first |
| Modern but weak engine | Modern score above the group's median and engine score below 50, ranked by how far apart they are |
| Strong engine but few modern features | The reverse |
| Best / worst value models | Average difference from fair price across the model's variants in the group |
| Model with most feature-loaded variants | How many of its variants are in the group's best-equipped quarter |
| Model with most high-safety variants | How many of its variants are in the group's safest quarter |
| Best engine models | Average engine score of the model's variants |
| Most feature-rich models | Average feature score of the model's variants |
| Picks among the priciest / cheapest variants | Best value within the group's most expensive / cheapest quarter |
| Where a gearbox, ADAS or 4x4/AWD starts | Price, cheapest first |
| Best value brand for 5 / 7 seaters | Average difference from fair price of the brand's 5-seat / 7-seat variants, for brands with at least 5 of them |

In [19]:
def lakh(rupees):
    """1534000 -> '₹15.34 lakh'; 12500000 -> '₹1.25 crore'."""
    if pd.isna(rupees):
        return "price not listed"
    if rupees >= 1e7:
        return f"₹{rupees / 1e7:.2f} crore"
    return f"₹{rupees / 1e5:.2f} lakh"

def gap_text(difference):
    """-0.123 -> '12% below', 0.081 -> '8% above', in whole percent cut toward zero.
    Cutting toward zero means a car 15.6% below reads '15% below' and is still "Good value",
    and one 14.8% below reads '14% below' and is "Fair": the words never contradict the badge."""
    if pd.isna(difference):
        return "with no fair price for"
    whole = int(abs(difference) * 100)
    if whole < 1:
        return "right at"
    return f"{whole}% {'below' if difference < 0 else 'above'}"

def gap_metric(difference):
    text = gap_text(difference)
    return "at fair price" if text == "right at" else f"{text} fair price"

def join(items):
    """['a', 'b', 'c'] -> 'a, b and c'."""
    items = [item for item in items if item]
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]

def plural(count, word, many=None):
    return f"{count} {word if count == 1 else (many or word + 's')}"

def split_list(text):
    """'a; b; c' -> ['a', 'b', 'c'], dropping CarDekho's '(stands for ...)' notes."""
    if pd.isna(text) or not str(text).strip():
        return []
    return [part.split(" (stands for")[0].strip() for part in str(text).split(";") if part.strip()]

AUTO_PHRASE = {"AMT": "an AMT", "CVT": "a CVT", "DCT": "a DCT", "Torque converter": "a torque-converter",
               "Hybrid": "a hybrid", "Automatic (type not listed)": "an", "Single-speed (electric)": "a single-speed electric",
               "Manual": "a manual"}

def gearbox_phrase(r):
    """'a DCT automatic', 'a manual gearbox' ..."""
    if r.gearbox == "Manual":
        return "a manual gearbox"
    if r.gearbox == "Single-speed (electric)":
        return "a single-speed electric drive"
    return f"{AUTO_PHRASE[r.gearbox]} automatic"

def first_upper(text):
    return text[:1].upper() + text[1:]

def share(k, n, noun="variant"):
    """('its only variant' | 'both of its variants' | 'all 4 of its variants' | '3 of its 9 variants', singular?)"""
    if n == 1:
        return f"its only {noun}", True
    if k == n:
        return (f"both of its {noun}s" if n == 2 else f"all {n} of its {noun}s"), False
    return f"{k} of its {n} {noun}s", k == 1

def rated(k, badge):
    """'none of them is rated Good value', '1 of them is ...', '3 of them are ...'."""
    if k == 0:
        return f"none of them is rated {badge}"
    return f"{k} of them {'is' if k == 1 else 'are'} rated {badge}"

def engine_place(pct, among):
    """Where an engine score sits, in words."""
    if pct <= 0.5:
        return f"the weakest {among}"
    if pct >= 99.5:
        return f"the strongest {among}"
    return f"stronger than {pct:.0f}% of SUVs {among}"

def fuel_word(r):
    """'petrol', 'diesel', 'CNG', 'petrol mild hybrid' for use mid-sentence."""
    return r.fuel if r.fuel.startswith("CNG") else r.fuel.lower()

def crash_phrase(r):
    if not r.crash_test:
        return "no crash-test rating yet"
    return f"a {int(r.crash_stars)}-star {r.crash_test} rating"

def highlights(r, limit=3):
    """A few things a buyer notices first, for the value-for-money reasons."""
    found = []
    if r.adas:
        found.append("ADAS")
    if r.crash_stars == 5:
        found.append(f"a 5-star {r.crash_test} rating")
    if r.airbags >= 6:
        found.append(f"{int(r.airbags)} airbags")
    if r.sunroof == "Panoramic":
        found.append("a panoramic sunroof")
    elif r.has_sunroof == 1:
        found.append("a sunroof")
    if r.has_ventilated_seats == 1:
        found.append("ventilated seats")
    if r.is_automatic:
        found.append(gearbox_phrase(r))
    if r.has_360_view_camera == 1:
        found.append("a 360° camera")
    return found[:limit]

def value_reason(r):
    text = f"{lakh(r.ex_showroom_price)} is {gap_text(r.difference_pct)} its fair price of {lakh(r.fair_price)}"
    if r.difference_pct > 0.005:
        text += f", so you pay about {lakh(r.ex_showroom_price - r.fair_price)} more than these features usually cost"
    elif r.difference_pct < -0.005:
        text += f", so you save about {lakh(r.fair_price - r.ex_showroom_price)} on what these features usually cost"
    text += "."
    found = highlights(r)
    if found:
        text += f" It has {join(found)}."
    return text

print(lakh(899000), "|", lakh(12500000), "|", gap_text(-0.156), "|", gap_text(-0.148), "|", gap_text(0.004), "|", join(["a", "b", "c"]))

₹8.99 lakh | ₹1.25 crore | 15% below | 14% below | right at | a, b and c


In [20]:
class Group:
    """One segment, one brand, or all SUVs, with the in-group scores the questions need."""

    def __init__(self, tab, name, frame, among, where, pool):
        self.tab, self.name, self.pool = tab, name, pool   # pool: "this segment", "this brand's SUVs", "all SUVs"

        self.among = among     # "in this segment", "from this brand", "in the market"
        self.where = where     # " in this segment", "" ...: used after "its 4 variants"
        g = frame.copy()
        g["engine_pct"] = engine_score(g)
        g["modern_pct"] = percentile(g["modern_score"])
        self.cars = g
        self.modern_median = g["modern_score"].median()
        self.feature_cut = g["feature_score"].quantile(0.75)
        self.safety_cut = g["safety_score"].quantile(0.75)

    def models(self):
        """One row per model, from its variants in this group."""
        g = self.cars.assign(
            loaded=self.cars["feature_score"] >= self.feature_cut,
            high_safety=self.cars["safety_score"] >= self.safety_cut,
            good=self.cars["badge"].eq("Good value"),
            over=self.cars["badge"].eq("Overpriced"),
        )
        return (
            g.groupby("model_name")
            .agg(company=("company", "first"), model=("model", "first"), variants=("name", "size"),
                 from_price=("ex_showroom_price", "min"), priced=("fair_price", "count"),
                 mean_difference=("difference_pct", "mean"), good=("good", "sum"), over=("over", "sum"),
                 mean_features=("features_present", "mean"), mean_feature_score=("feature_score", "mean"),
                 loaded=("loaded", "sum"), high_safety=("high_safety", "sum"), mean_safety=("safety_score", "mean"),
                 mean_engine=("engine_pct", "mean"), mean_bhp=("power_bhp", "mean"), mean_torque=("torque_nm", "mean"),
                 mean_modern=("modern_score", "mean"), mean_modern_pct=("modern_pct", "mean"))
            .reset_index()
        )


def take(candidates, n):
    """The first n, or only the first one when fewer than n qualify (the small-group rule)."""
    if n == 0 or len(candidates) >= n:
        return (candidates if n == 0 else candidates.head(n)), ""
    if len(candidates) == 0:
        return candidates, ""
    if len(candidates) == 1:
        return candidates, "Only 1 qualifies here."
    return candidates.head(1), f"Only {len(candidates)} qualify here, fewer than the {n} asked for, so only the top one is shown."


def variant_row(r, rank, metric, reason, highlight=""):
    return dict(rank=rank, item_type="variant", name=r.name, company=r.company, model=r.model, variant=r.variant,
                price=r.ex_showroom_price, metric=metric, highlight=highlight, reason=reason)

def model_row(m, rank, metric, reason, price, highlight=""):
    return dict(rank=rank, item_type="model", name=m.model_name, company=m.company, model=m.model, variant="",
                price=price, metric=metric, highlight=highlight, reason=reason)

def note_row(text):
    return dict(rank=0, item_type="note", name="", company="", model="", variant="", price=np.nan,
                metric="", highlight="", reason=text)

def finish(rows, note="", empty=""):
    """Put the note first; when nothing qualifies, return just the 'empty' note."""
    if not rows:
        return [note_row(empty)] if empty else []
    return ([note_row(note)] if note else []) + rows

The functions for the variant questions: value, features, safety, and modern features against engine.

In [21]:
def k_value_top(G, q):
    c = G.cars[G.cars["fair_price"].notna()].sort_values(["difference_pct", "ex_showroom_price", "name"])
    shown, note = take(c, q.list_size)
    rows = [variant_row(r, i, gap_metric(r.difference_pct), value_reason(r)) for i, r in enumerate(shown.itertuples(), 1)]
    return finish(rows, note, empty=f"No variant {G.among} has a fair price to compare with.")

def k_value_bottom(G, q):
    c = G.cars[G.cars["fair_price"].notna()].sort_values(["difference_pct", "ex_showroom_price", "name"], ascending=[False, False, True])
    shown, note = take(c, q.list_size)
    rows = []
    for i, r in enumerate(shown.itertuples(), 1):
        reason = value_reason(r)
        peers = split_list(r.missing_vs_price_peers)
        if peers:
            reason += f" It lacks what most SUVs at this price have: {', '.join(peers[:3])}."
        rows.append(variant_row(r, i, gap_metric(r.difference_pct), reason))
    return finish(rows, note, empty=f"No variant {G.among} has a fair price to compare with.")

def feature_reason(r):
    text = f"Has {r.features_present} of the {N_FEATURES} listed features"
    missing = split_list(r.basics_missing)
    if missing:
        text += f", but lacks {join(missing)} ({'a basic feature' if len(missing) == 1 else 'basic features'})"
    else:
        text += ", and every basic feature"
    extras = split_list(r.modern_list)[:4] or split_list(r.feature_list)[-4:]
    if extras:
        text += f". Includes {join(extras)}"
    return text + "."

def k_features_top(G, q):
    c = G.cars.sort_values(["feature_score", "difference_pct", "ex_showroom_price", "name"], ascending=[False, True, True, True])
    shown, note = take(c, q.list_size)
    top3 = shown.head(3)
    priced3 = top3[top3["fair_price"].notna()]
    best_name = priced3.sort_values("difference_pct")["name"].iloc[0] if len(top3) > 1 and len(priced3) else None
    rows = []
    for i, r in enumerate(shown.itertuples(), 1):
        reason, highlight = feature_reason(r), ""
        if r.name == best_name:
            others = [gap_text(d) for n, d in zip(top3["name"], top3["difference_pct"]) if n != r.name]
            reason += (f" Of the top 3, it is the best value for money: {gap_text(r.difference_pct)} its fair price, "
                       f"while the other {'two are' if len(others) > 1 else 'one is'} {join(others)} theirs.")
            highlight = "Best value of the top 3"
        rows.append(variant_row(r, i, f"{r.features_present} of {N_FEATURES} features", reason, highlight))
    return finish(rows, note)

def crash_sentence(r):
    if not r.crash_test:
        return "No crash-test rating yet."
    effect = {10: "adds 10 points", 5: "adds 5 points", 0: "neither adds nor takes off points",
              -10: "takes off 10 points"}[int(r.crash_points)]
    return f"Its {int(r.crash_stars)}-star {r.crash_test} rating {effect}."

def safety_reason(r, worst=False):
    has = split_list(r.safety_list) + ([r.camera] if r.camera else [])
    lacks = split_list(r.safety_missing) + ([] if r.camera else ["a rear camera"])
    airbag_text = f"{int(r.airbags)} airbags" if pd.notna(r.airbags) else "airbags not listed"
    if worst and pd.notna(r.airbags) and r.airbags <= 4:
        airbag_text = "only " + airbag_text
    text = f"Safety score {r.safety_score:.0f}: {join([airbag_text] + has)} (equipment {r.safety_equipment:.0f} of 100)."
    if worst and lacks:
        text += f" Not listed: {', '.join(lacks)}."
    return f"{text} {crash_sentence(r)}"

def k_safety_top(G, q):
    c = G.cars.sort_values(["safety_score", "ex_showroom_price", "name"], ascending=[False, True, True])
    shown, note = take(c, q.list_size)
    rows = [variant_row(r, i, f"safety {r.safety_score:.0f}", safety_reason(r)) for i, r in enumerate(shown.itertuples(), 1)]
    return finish(rows, note)

def k_safety_bottom(G, q):
    c = G.cars.sort_values(["safety_score", "ex_showroom_price", "name"], ascending=[True, False, True])
    shown, note = take(c, q.list_size)
    rows = [variant_row(r, i, f"safety {r.safety_score:.0f}", safety_reason(r, worst=True)) for i, r in enumerate(shown.itertuples(), 1)]
    return finish(rows, note)

def engine_text(r):
    return f"{r.power_bhp:.0f} bhp and {r.torque_nm:.0f} Nm" if pd.notna(r.power_bhp) and pd.notna(r.torque_nm) else "power or torque not listed"

def k_modern_weak_engine(G, q):
    c = G.cars.dropna(subset=["engine_pct"])
    c = c[(c["modern_score"] > G.modern_median) & (c["engine_pct"] < 50)]
    c = c.assign(gap=c["modern_pct"] - c["engine_pct"]).sort_values(["gap", "ex_showroom_price", "name"], ascending=[False, True, True])
    shown, note = take(c, q.list_size)
    rows = []
    for i, r in enumerate(shown.itertuples(), 1):
        modern = split_list(r.modern_list)
        listed = join(modern[:4] + ["more"]) if len(modern) > 4 else join(modern)
        weak = "is the weakest of them" if r.engine_pct <= 0.5 else f"is stronger than only {r.engine_pct:.0f}% of them"
        reason = (f"{r.modern_score} of the {N_MODERN} eye-catching features ({listed}), "
                  f"more than most SUVs {G.among}, but its engine ({engine_text(r)}) {weak}.")
        rows.append(variant_row(r, i, f"modern {r.modern_score}/{N_MODERN}, engine {r.engine_pct:.0f}", reason))
    return finish(rows, note, empty=f"No variant {G.among} is both above the middle on modern features and below the middle on engine performance.")

def k_engine_weak_modern(G, q):
    c = G.cars.dropna(subset=["engine_pct"])
    c = c[(c["engine_pct"] >= 50) & (c["modern_score"] < G.modern_median)]
    c = c.assign(gap=c["engine_pct"] - c["modern_pct"]).sort_values(["gap", "ex_showroom_price", "name"], ascending=[False, True, True])
    shown, note = take(c, q.list_size)
    rows = []
    for i, r in enumerate(shown.itertuples(), 1):
        few = "none of the" if r.modern_score == 0 else f"only {r.modern_score} of the"
        reason = (f"{engine_text(r)}: {engine_place(r.engine_pct, G.among)}, "
                  f"but {few} {N_MODERN} eye-catching features, fewer than most of them.")
        rows.append(variant_row(r, i, f"engine {r.engine_pct:.0f}, modern {r.modern_score}/{N_MODERN}", reason))
    return finish(rows, note, empty=f"No variant {G.among} is both above the middle on engine performance and below the middle on modern features.")

The model-level questions. Each model is judged only on its variants **in the group**, so a model's ₹10–15 lakh variants decide its place in that segment, not its pricier versions elsewhere.

In [22]:
def variants_of(G, model_name):
    return G.cars[G.cars["model_name"] == model_name]

def k_model_value_top(G, q, worst=False):
    t = G.models()
    t = t[t["priced"] > 0].sort_values(["mean_difference", "from_price"], ascending=[not worst, True])
    shown, note = take(t, q.list_size)
    rows = []
    for i, m in enumerate(shown.itertuples(), 1):
        mine = variants_of(G, m.model_name).dropna(subset=["fair_price"]).sort_values("difference_pct", ascending=not worst)
        pick = mine.iloc[0]
        if m.priced == 1:
            reason = f"Its only variant{G.where} with a fair price, the {pick['variant']}, is {gap_text(pick['difference_pct'])} its fair price."
        else:
            badge_count = m.over if worst else m.good
            badge_word = "Overpriced" if worst else "Good value"
            reason = (f"Its {m.priced} variants{G.where} average {gap_text(m.mean_difference)} their fair prices, "
                      f"and {rated(badge_count, badge_word)}. "
                      f"{'Worst' if worst else 'Best'}: the {pick['variant']}, {gap_text(pick['difference_pct'])} its fair price.")
        rows.append(model_row(m, i, f"average {gap_metric(m.mean_difference)}", reason, m.from_price))
    return finish(rows, note)

def k_model_value_bottom(G, q):
    return k_model_value_top(G, q, worst=True)

def k_model_most_loaded(G, q):
    t = G.models()
    t = t[t["loaded"] > 0].assign(share=lambda d: d["loaded"] / d["variants"])
    t = t.sort_values(["loaded", "share", "mean_feature_score"], ascending=False)
    shown, note = take(t, q.list_size)
    rows = []
    for i, m in enumerate(shown.itertuples(), 1):
        text, single = share(m.loaded, m.variants)
        rows.append(model_row(m, i, f"{m.loaded} of {m.variants} variants",
                              f"{first_upper(text)}{G.where} {'is' if single else 'are'} in the best-equipped quarter of {G.pool} "
                              f"(a feature score of {G.feature_cut:.0f} or more, out of {N_FEATURES}).", m.from_price))
    return finish(rows, note)

def k_model_most_safe(G, q):
    t = G.models()
    t = t[t["high_safety"] > 0].assign(share=lambda d: d["high_safety"] / d["variants"])
    t = t.sort_values(["high_safety", "share", "mean_safety"], ascending=False)
    shown, note = take(t, q.list_size)
    rows = []
    for i, m in enumerate(shown.itertuples(), 1):
        text, single = share(m.high_safety, m.variants)
        rows.append(model_row(m, i, f"{m.high_safety} of {m.variants} variants",
                              f"{first_upper(text)}{G.where} {'is' if single else 'are'} in the safest quarter of {G.pool} "
                              f"(a safety score of {G.safety_cut:.0f} or more); its variants average {m.mean_safety:.0f}.", m.from_price))
    return finish(rows, note)

def k_model_engine_top(G, q):
    t = G.models().dropna(subset=["mean_engine"]).sort_values(["mean_engine", "from_price"], ascending=[False, True])
    shown, note = take(t, q.list_size)
    only_model = G.cars["model_name"].nunique() == 1
    rows = []
    for i, m in enumerate(shown.itertuples(), 1):
        power = f"{m.mean_bhp:.0f} bhp" + (f" and {m.mean_torque:.0f} Nm" if pd.notna(m.mean_torque) else "")
        if only_model:
            reason = f"It is the only model {G.among}; its {plural(m.variants, 'variant')} average {power}."
        else:
            reason = f"Its {plural(m.variants, 'variant')}{G.where} average {power}: on average {engine_place(m.mean_engine, G.among)}."
        rows.append(model_row(m, i, f"engine score {m.mean_engine:.0f}", reason, m.from_price))
    return finish(rows, note)

def k_model_modern_weak_engine(G, q):
    t = G.models().dropna(subset=["mean_engine"])
    t = t[(t["mean_modern"] > G.modern_median) & (t["mean_engine"] < 50)]
    t = t.assign(gap=t["mean_modern_pct"] - t["mean_engine"]).sort_values(["gap", "from_price"], ascending=[False, True])
    shown, note = take(t, q.list_size)
    rows = []
    for i, m in enumerate(shown.itertuples(), 1):
        power = f"{m.mean_bhp:.0f} bhp" + (f" and {m.mean_torque:.0f} Nm" if pd.notna(m.mean_torque) else "")
        weak = "the weakest of them" if m.mean_engine <= 0.5 else f"stronger than only {m.mean_engine:.0f}% of them"
        if m.variants == 1:
            reason = (f"Its only variant{G.where} has {m.mean_modern:.0f} of the {N_MODERN} eye-catching features, more than most SUVs "
                      f"{G.among}, but its engine ({power}) is {weak}.")
        else:
            reason = (f"Its {m.variants} variants{G.where} average {m.mean_modern:.1f} of the {N_MODERN} eye-catching features, "
                      f"more than most SUVs {G.among}, but their engines average {power}, {weak}.")
        rows.append(model_row(m, i, f"modern {m.mean_modern:.1f}/{N_MODERN}, engine {m.mean_engine:.0f}", reason, m.from_price))
    return finish(rows, note, empty=f"No model {G.among} is both above the middle on modern features and below the middle on engine performance.")

def k_model_features_top(G, q):
    t = G.models().sort_values(["mean_feature_score", "from_price"], ascending=[False, True])
    shown, note = take(t, q.list_size)
    rows = []
    for i, m in enumerate(shown.itertuples(), 1):
        best = variants_of(G, m.model_name).sort_values(["feature_score", "ex_showroom_price"], ascending=[False, True]).iloc[0]
        reason = (f"Its {plural(m.variants, 'variant')}{G.where} average {m.mean_features:.1f} of the {N_FEATURES} listed features "
                  f"(feature score {m.mean_feature_score:.1f}). Best equipped: the {best['variant']}, with {best['features_present']} features.")
        rows.append(model_row(m, i, f"average {m.mean_features:.1f} of {N_FEATURES} features", reason, m.from_price))
    return finish(rows, note)

Buying at the top or the bottom of a group's price range. "The most expensive variants" means the group's **priciest quarter** by price, and "the least expensive" its **cheapest quarter**. Within that range, the two picks are the ones with the best value for money: the most features for the money, according to the fair price.

In [23]:
def k_price_picks(G, q, top):
    cut = G.cars["ex_showroom_price"].quantile(0.75 if top else 0.25)
    pool = G.cars[G.cars["ex_showroom_price"] >= cut] if top else G.cars[G.cars["ex_showroom_price"] <= cut]
    c = pool[pool["fair_price"].notna()].sort_values(["difference_pct", "ex_showroom_price", "name"])
    shown, note = take(c, q.list_size)
    which = "priciest" if top else "cheapest"
    bound = f"{lakh(cut)} and up" if top else f"up to {lakh(cut)}"
    lead = (f"The only variant in the {which} quarter {G.among} ({bound})" if len(pool) == 1
            else f"Best value among the {len(pool)} {which} variants {G.among} ({bound})")
    rows = [variant_row(r, i, gap_metric(r.difference_pct), f"{lead}: {value_reason(r)}")
            for i, r in enumerate(shown.itertuples(), 1)]
    summary = f"The {which} quarter {G.among} is {bound}: {plural(len(pool), 'variant')}."
    return finish(rows, " ".join(x for x in [summary, note] if x))

def k_pricey_picks(G, q):
    return k_price_picks(G, q, top=True)

def k_cheap_picks(G, q):
    return k_price_picks(G, q, top=False)

Where a gearbox, ADAS or 4x4/AWD starts, the 7-seaters, and the engine sizes.

In [24]:
def k_auto_start(G, q):
    c = G.cars[G.cars["is_automatic"]].sort_values(["ex_showroom_price", "name"])
    if c.empty:
        return [note_row(f"No automatic (non-electric) variant {G.among}.")]
    shown, note = take(c, q.list_size)
    first = c.iloc[0]
    kind_text = "type not listed" if first["gearbox"] == "Automatic (type not listed)" else first["gearbox"]
    summary = f"Automatic gearboxes start at {lakh(first['ex_showroom_price'])} {G.among}, with the {first['name']} ({kind_text})."
    unlisted = (G.cars["gearbox"] == "Automatic (type not listed)").sum()
    if unlisted:
        summary += f" {plural(unlisted, 'automatic')} here {'doesn' if unlisted == 1 else 'don'}'t say which type {'it is' if unlisted == 1 else 'they are'}."
    if (G.cars["fuel_type"] == "Electric").any():
        summary += " Electric cars are left out: they have no gearbox."
    rows = [variant_row(r, i, r.gearbox, f"{first_upper(gearbox_phrase(r))}, {fuel_word(r)}, {r.power_bhp:.0f} bhp.")
            for i, r in enumerate(shown.itertuples(), 1)]
    return finish(rows, " ".join(x for x in [summary, note] if x))

def k_gearbox_start(G, q, gearbox):
    c = G.cars[G.cars["gearbox"] == gearbox].sort_values(["ex_showroom_price", "name"])
    word = "torque-converter automatics" if gearbox == "Torque converter" else f"{gearbox} automatics"
    if c.empty:
        return [note_row(f"No {word[:-1]} {G.among}.")]
    shown, note = take(c, q.list_size)
    first = c.iloc[0]
    summary = f"{word[0].upper() + word[1:]} start at {lakh(first['ex_showroom_price'])} {G.among}, with the {first['name']}."
    if gearbox == "CVT" and (G.cars["gearbox"] == "Hybrid").any():
        summary += " Strong hybrids use their own hybrid gearbox, so they aren't counted as CVTs."
    rows = []
    for i, r in enumerate(shown.itertuples(), 1):
        gears = f"{int(r.gears)}-speed " if pd.notna(r.gears) and gearbox != "CVT" else ""
        rows.append(variant_row(r, i, gearbox, f"{gears}{gearbox if gearbox != 'Torque converter' else 'torque-converter automatic'}, {fuel_word(r)}, {r.power_bhp:.0f} bhp."))
    return finish(rows, " ".join(x for x in [summary, note] if x))

def k_drive(G, q):
    rows, notes = [], []
    for kind in ["4x4", "AWD"]:
        with_kind = G.cars[G.cars["drive_type"] == kind]
        if with_kind.empty:
            notes.append(f"No {kind} variants {G.among}.")
            continue
        per_model = with_kind.sort_values("ex_showroom_price").groupby("model_name", sort=False)
        notes.append(f"{plural(per_model.ngroups, 'model')} {'offers' if per_model.ngroups == 1 else 'offer'} {kind} {G.among}, from {lakh(with_kind['ex_showroom_price'].min())}.")
        for model_name, grp in per_model:
            total = len(variants_of(G, model_name))
            cheapest = grp.iloc[0]
            text, single = share(len(grp), total)
            has_kind = f"{first_upper(text)}{G.where} {'has' if single else 'have'} {kind}"
            rows.append(dict(rank=len(rows) + 1, item_type="model", name=model_name, company=cheapest["company"], model=cheapest["model"], variant="",
                             price=cheapest["ex_showroom_price"], metric=kind, highlight=kind,
                             reason=f"{has_kind}, from {lakh(cheapest['ex_showroom_price'])} (the {cheapest['variant']})."))
    return [note_row(" ".join(notes))] + rows

def k_seven_seaters(G, q):
    sevens = G.cars[G.cars["seven_plus"]]
    if sevens.empty:
        return [note_row(f"No 7-seater variants {G.among}.")]
    rows = []
    for model_name, grp in sevens.sort_values("ex_showroom_price").groupby("model_name", sort=False):
        mine = variants_of(G, model_name)
        cheapest = grp.iloc[0]
        seat_counts = join([str(s) for s in sorted(grp["seats"].unique())])
        text, single = share(len(grp), len(mine))
        if len(grp) == len(mine):
            reason, metric = f"{first_upper(text)}{G.where} {'has' if single else 'have'} {seat_counts} seats.", "All variants"
        else:
            others = join([str(s) for s in sorted(mine.loc[~mine["seven_plus"], "seats"].unique())])
            reason, metric = (f"{first_upper(text)}{G.where} {'has' if single else 'have'} {seat_counts} seats; the others have {others}.",
                              f"Some ({len(grp)} of {len(mine)})")
        reason += f" Cheapest 7-seater: the {cheapest['variant']}, at {lakh(cheapest['ex_showroom_price'])}."
        rows.append(dict(rank=len(rows) + 1, item_type="model", name=model_name, company=cheapest["company"], model=cheapest["model"],
                         variant="", price=cheapest["ex_showroom_price"], metric=metric, highlight="", reason=reason))
    all_count = sum(r["metric"] == "All variants" for r in rows)
    if len(rows) == 1:
        tail = "it only comes as a 7-seater." if all_count else "it also comes with fewer seats."
    else:
        tail = f"{all_count} of them only {'comes' if all_count == 1 else 'come'} as 7-seaters."
    summary = f"{plural(len(rows), 'model')} {G.among} {'has' if len(rows) == 1 else 'have'} 7-seater variants; {tail}"
    return [note_row(summary)] + rows

def adas_reason(r):
    aeb = " with automatic emergency braking" if r.has_automatic_emergency_braking == 1 else ""
    airbag_text = f"{int(r.airbags)} airbags" if pd.notna(r.airbags) else "airbags not listed"
    return f"Has ADAS{aeb}, {airbag_text} and {gearbox_phrase(r)}."

def k_adas_cheapest(G, q):
    c = G.cars[G.cars["adas"]].sort_values(["ex_showroom_price", "name"])
    if c.empty:
        return [note_row(f"No variant with ADAS {G.among}.")]
    shown, note = take(c, q.list_size)
    summary = f"ADAS starts at {lakh(c['ex_showroom_price'].iloc[0])} {G.among}, with the {c['name'].iloc[0]}."
    rows = [variant_row(r, i, "ADAS with AEB" if r.has_automatic_emergency_braking == 1 else "ADAS", adas_reason(r))
            for i, r in enumerate(shown.itertuples(), 1)]
    return finish(rows, " ".join(x for x in [summary, note] if x))

def k_adas_value(G, q):
    c = G.cars[G.cars["adas"] & G.cars["fair_price"].notna()].sort_values(["difference_pct", "ex_showroom_price", "name"])
    if c.empty:
        return [note_row(f"No variant with ADAS {G.among}.")]
    shown, note = take(c, q.list_size)
    rows = [variant_row(r, i, gap_metric(r.difference_pct), value_reason(r)) for i, r in enumerate(shown.itertuples(), 1)]
    return finish(rows, note)

def battery_range(evs):
    low, high = evs["battery_kwh"].min(), evs["battery_kwh"].max()
    return f"a {low:g} kWh battery" if low == high else f"batteries of {low:g}–{high:g} kWh"

def k_engine_sizes(G, q):
    c = G.cars
    fuel_cars = c[c["fuel_type"] != "Electric"]
    parts = []
    if not fuel_cars.empty:
        low, high = fuel_cars["engine_cc"].min(), fuel_cars["engine_cc"].max()
        parts.append(f"All engines {G.among} are {low:,.0f} cc." if low == high else f"Engines {G.among} range from {low:,.0f} cc to {high:,.0f} cc.")
    evs = c[c["fuel_type"] == "Electric"]
    if not evs.empty:
        parts.append(f"{plural(len(evs), 'electric variant')} {'has' if len(evs) == 1 else 'have'} no engine, with "
                     f"{battery_range(evs)}.")
    order = sorted(c["engine_size"].unique(), key=lambda s: (s == "Electric", s))
    rows = []
    for size in order:
        grp = c[c["engine_size"] == size].sort_values(["ex_showroom_price", "name"])
        first = grp.iloc[0]
        if size == "Electric":
            detail = f"{plural(len(grp), 'electric variant')}, with {battery_range(grp)}"
        else:
            low, high = grp["engine_cc"].min(), grp["engine_cc"].max()
            detail = f"{plural(len(grp), 'variant')}, " + (f"{low:,.0f} cc" if low == high else f"{low:,.0f}–{high:,.0f} cc")
        rows.append(dict(rank=len(rows) + 1, item_type="engine size", name=size, company=first["company"], model=first["model"],
                         variant=first["variant"], price=first["ex_showroom_price"], metric=f"from {lakh(first['ex_showroom_price'])}",
                         highlight="", reason=f"{detail}. Cheapest: the {first['name']}, at {lakh(first['ex_showroom_price'])}."))
    return [note_row(" ".join(parts))] + rows

The questions asked only for all SUVs together: the cheapest models with ADAS, the best-value brands for 5 and 7 seaters, and where 4x4 or AWD starts.

For the brand questions, a brand needs at least **5** variants with that seat count, so that one or two lucky cars can't decide the answer. The question asks "which brand", so rank 1 is the answer, and the runners-up are shown for context.

In [25]:
MIN_BRAND_VARIANTS = 5

def k_models_adas_cheapest(G, q):
    c = G.cars[G.cars["adas"]].sort_values(["ex_showroom_price", "name"])
    first_per_model = c.drop_duplicates("model_name")
    shown, note = take(first_per_model, q.list_size)
    rows = []
    for i, r in enumerate(shown.itertuples(), 1):
        mine = variants_of(G, r.model_name)
        with_adas = int(mine["adas"].sum())
        reason = (f"Its cheapest variant with ADAS is the {r.variant}, at {lakh(r.ex_showroom_price)}; "
                  f"{with_adas} of its {len(mine)} variants {'has' if with_adas == 1 else 'have'} ADAS.")
        rows.append(dict(rank=i, item_type="model", name=r.model_name, company=r.company, model=r.model, variant="",
                         price=r.ex_showroom_price, metric=f"{with_adas} of {len(mine)} variants with ADAS", highlight="", reason=reason))
    return finish(rows, note)

def k_brand_value(G, q, seats):
    label = "five-seat" if seats == "5" else "seven-seat"
    c = G.cars[G.cars["fair_price"].notna() & (G.cars["seats"].eq(5) if seats == "5" else G.cars["seven_plus"])]
    t = (c.assign(good=c["badge"].eq("Good value"))
         .groupby("company").agg(n=("name", "size"), mean_difference=("difference_pct", "mean"), good=("good", "sum"))
         .reset_index())
    left_out = sorted(t.loc[t["n"] < MIN_BRAND_VARIANTS, "company"])
    t = t[t["n"] >= MIN_BRAND_VARIANTS].sort_values(["mean_difference", "n"], ascending=[True, False])
    shown, note = take(t, q.list_size)
    rows = []
    for i, b in enumerate(shown.itertuples(), 1):
        best = c[c["company"] == b.company].sort_values("difference_pct").iloc[0]
        reason = (f"Its {b.n} {label} variants average {gap_text(b.mean_difference)} their fair prices, and "
                  f"{rated(b.good, 'Good value')}. Best: the {best['name']}, {gap_text(best['difference_pct'])} its fair price.")
        rows.append(dict(rank=i, item_type="brand", name=b.company, company=b.company, model="", variant="", price=np.nan,
                         metric=f"average {gap_metric(b.mean_difference)}", highlight="Answer" if i == 1 else "", reason=reason))
    summary = (f"{len(t)} brands have at least {MIN_BRAND_VARIANTS} {label} variants with a fair price"
               + (f"; left out for having fewer: {join(left_out)}." if left_out else "."))
    return finish(rows, " ".join(x for x in [summary, note] if x))

def k_drive_start(G, q):
    c = G.cars[G.cars["drive_type"].isin(["4x4", "AWD"])].sort_values(["ex_showroom_price", "name"])
    shown, note = take(c, q.list_size)
    first = c.iloc[0]
    summary = f"4x4 or AWD starts at {lakh(first['ex_showroom_price'])}, with the {first['name']} ({first['drive_type']})."
    rows = [variant_row(r, i, r.drive_type, f"{r.drive_type}, {fuel_word(r)}, {gearbox_phrase(r)}, {r.power_bhp:.0f} bhp.")
            for i, r in enumerate(shown.itertuples(), 1)]
    return finish(rows, " ".join(x for x in [summary, note] if x))

KINDS = {
    "value_top": k_value_top, "value_bottom": k_value_bottom, "features_top": k_features_top,
    "safety_top": k_safety_top, "safety_bottom": k_safety_bottom, "drive": k_drive,
    "modern_weak_engine": k_modern_weak_engine, "engine_weak_modern": k_engine_weak_modern,
    "model_value_top": k_model_value_top, "model_value_bottom": k_model_value_bottom,
    "model_most_loaded": k_model_most_loaded, "model_engine_top": k_model_engine_top,
    "model_modern_weak_engine": k_model_modern_weak_engine, "model_features_top": k_model_features_top,
    "pricey_picks": k_pricey_picks, "cheap_picks": k_cheap_picks, "model_most_safe": k_model_most_safe,
    "auto_start": k_auto_start, "seven_seaters": k_seven_seaters, "adas_cheapest": k_adas_cheapest,
    "adas_value": k_adas_value, "engine_sizes": k_engine_sizes, "models_adas_cheapest": k_models_adas_cheapest,
    "drive_start": k_drive_start,
}

def run_question(G, q):
    kind, _, param = q.kind.partition(":")
    if kind == "gearbox_start":
        return k_gearbox_start(G, q, param)
    if kind == "brand_value":
        return k_brand_value(G, q, param)
    return KINDS[kind](G, q)

ANSWER_COLUMNS = ["tab", "group", "question_id", "rank", "item_type", "name", "company", "model", "variant",
                  "price", "metric", "highlight", "reason"]

def answer_group(G):
    rows = []
    for q in questions[questions["tab"] == G.tab].itertuples():
        for row in run_question(G, q):
            rows.append({"tab": G.tab, "group": G.name, "question_id": q.question_id, **row})
    return rows

def show(answers, tab, group):
    """Print every answer for one group, question by question."""
    block = answers[(answers["tab"] == tab) & (answers["group"] == group)]
    wrap = dict(width=118, initial_indent="          ", subsequent_indent="          ")
    for category, qs in questions[questions["tab"] == tab].groupby("category", sort=False):
        print(f"\n▌ {category.upper()}")
        for q in qs.itertuples():
            print(f"\n  {q.question_id}  {q.question}")
            for r in block[block["question_id"] == q.question_id].itertuples():
                if r.item_type == "note":
                    print(textwrap.fill("• " + r.reason, width=118, initial_indent="      ", subsequent_indent="        "))
                    continue
                price = (f" — {lakh(r.price)}" if r.item_type == "variant" else f" — from {lakh(r.price)}" if r.item_type == "model" else "") if pd.notna(r.price) else ""
                tag = f"  [{r.highlight}]" if r.highlight and r.highlight not in r.metric else ""
                print(f"     {int(r.rank):>2}. {r.name}{price} — {r.metric}{tag}")
                print(textwrap.fill(r.reason, **wrap))

## Part 6: Segment insights

Every segment answers all 26 questions. Within a segment, engine and modern-feature comparisons are made against the other SUVs in that segment.

In [26]:
segment_groups = [
    Group("segment", s.segment, cars[cars["segment_id"] == s.segment_id], among="in this segment", where=" in this segment", pool="this segment")
    for s in SEGMENTS.itertuples()
]
segment_answers = pd.DataFrame([row for G in segment_groups for row in answer_group(G)], columns=ANSWER_COLUMNS)
print(f"{len(segment_answers):,} answer rows for {len(segment_groups)} segments "
      f"({(segment_answers['item_type'] != 'note').sum():,} ranked items and {(segment_answers['item_type'] == 'note').sum()} notes).")

def show_segment(segment_id):
    s = segments[segments["segment_id"] == segment_id].iloc[0]
    print("═" * 118)
    print(f"{s['segment']}: {s['variants']} variants of {s['models']} models from {s['brands']} brands, "
          f"{lakh(s['from_price'])} to {lakh(s['to_price'])}")
    print(f"Average price {lakh(s['avg_price'])}, standard deviation {lakh(s['std_price'])}")
    print("═" * 118)
    show(segment_answers, "segment", s["segment"])

1,014 answer rows for 7 segments (943 ranked items and 71 notes).


### Under ₹10 lakh

In [27]:
show_segment(1)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Under ₹10 lakh: 185 variants of 20 models from 11 brands, ₹5.69 lakh to ₹10.00 lakh
Average price ₹8.71 lakh, standard deviation ₹1.06 lakh
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▌ VALUE FOR MONEY

  S01  10 most value for money variants, in order
      1. Tata Punch Adventure Turbo — ₹8.47 lakh — 29% below fair price
          ₹8.47 lakh is 29% below its fair price of ₹11.97 lakh, so you save about ₹3.50 lakh on what these features
          usually cost. It has ADAS, a 5-star Bharat NCAP rating and 6 airbags.
      2. Tata Punch Smart — ₹5.75 lakh — 26% below fair price
          ₹5.75 lakh is 26% below its fair price of ₹7.83 lakh, so you save about ₹2.08 lakh on what these features
          usually cost. It has a 5-star Bharat NCAP rating and 6 airbags.
      3. Skoda Kylaq Classic — 

### ₹10–15 lakh

In [28]:
show_segment(2)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
₹10–15 lakh: 346 variants of 45 models from 13 brands, ₹10.09 lakh to ₹15.00 lakh
Average price ₹12.66 lakh, standard deviation ₹1.37 lakh
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▌ VALUE FOR MONEY

  S01  10 most value for money variants, in order
      1. Mahindra Scorpio N Z2 Petrol MT — ₹13.69 lakh — 26% below fair price
          ₹13.69 lakh is 26% below its fair price of ₹18.65 lakh, so you save about ₹4.96 lakh on what these features
          usually cost. It has a 5-star Global NCAP rating.
      2. Renault Duster Adventure 1.3 Turbo — ₹13.99 lakh — 21% below fair price
          ₹13.99 lakh is 21% below its fair price of ₹17.81 lakh, so you save about ₹3.82 lakh on what these features
          usually cost. It has a 5-star Bharat NCAP rating, 6 airbags and a panoramic sunroof.
   

### ₹15–20 lakh

In [29]:
show_segment(3)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
₹15–20 lakh: 363 variants of 39 models from 13 brands, ₹15.04 lakh to ₹20.00 lakh
Average price ₹17.70 lakh, standard deviation ₹1.50 lakh
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▌ VALUE FOR MONEY

  S01  10 most value for money variants, in order
      1. Renault Duster Adventure 1.3 Turbo DCT — ₹15.39 lakh — 20% below fair price
          ₹15.39 lakh is 20% below its fair price of ₹19.33 lakh, so you save about ₹3.94 lakh on what these features
          usually cost. It has a 5-star Bharat NCAP rating, 6 airbags and a panoramic sunroof.
      2. Mahindra Thar ROXX Star Edition Diesel AT — ₹19.48 lakh — 19% below fair price
          ₹19.48 lakh is 19% below its fair price of ₹24.11 lakh, so you save about ₹4.63 lakh on what these features
          usually cost. It has ADAS, a 5-star Bha

### ₹20–30 lakh

In [30]:
show_segment(4)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
₹20–30 lakh: 283 variants of 27 models from 9 brands, ₹20.06 lakh to ₹29.95 lakh
Average price ₹23.69 lakh, standard deviation ₹2.39 lakh
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▌ VALUE FOR MONEY

  S01  10 most value for money variants, in order
      1. Tata Harrier EV Fearless Plus QWD 75 — ₹26.79 lakh — 35% below fair price
          ₹26.79 lakh is 35% below its fair price of ₹41.53 lakh, so you save about ₹14.74 lakh on what these features
          usually cost. It has a 5-star Bharat NCAP rating, 7 airbags and a panoramic sunroof.
      2. Tata Harrier EV Fearless Plus QWD 75 ACFC — ₹27.29 lakh — 34% below fair price
          ₹27.29 lakh is 34% below its fair price of ₹41.53 lakh, so you save about ₹14.24 lakh on what these features
          usually cost. It has a 5-star Bharat NCA

### ₹30–50 lakh

In [31]:
show_segment(5)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
₹30–50 lakh: 79 variants of 22 models from 16 brands, ₹30.04 lakh to ₹50.00 lakh
Average price ₹38.15 lakh, standard deviation ₹5.54 lakh
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▌ VALUE FOR MONEY

  S01  10 most value for money variants, in order
      1. Tata Harrier EV Empowered QWD 75 Stealth — ₹30.04 lakh — 28% below fair price
          ₹30.04 lakh is 28% below its fair price of ₹41.80 lakh, so you save about ₹11.76 lakh on what these features
          usually cost. It has ADAS, a 5-star Bharat NCAP rating and 7 airbags.
      2. Tata Harrier EV Empowered QWD 75 Stealth ACFC — ₹30.54 lakh — 26% below fair price
          ₹30.54 lakh is 26% below its fair price of ₹41.80 lakh, so you save about ₹11.26 lakh on what these features
          usually cost. It has ADAS, a 5-star Bharat NCAP

### ₹50 lakh–1 crore

In [32]:
show_segment(6)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
₹50 lakh–1 crore: 51 variants of 31 models from 15 brands, ₹50.60 lakh to ₹99.85 lakh
Average price ₹70.46 lakh, standard deviation ₹14.51 lakh
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▌ VALUE FOR MONEY

  S01  10 most value for money variants, in order
      1. Volvo XC90 B5 AWD — ₹97.80 lakh — 41% below fair price
          ₹97.80 lakh is 41% below its fair price of ₹1.67 crore, so you save about ₹69.21 lakh on what these features
          usually cost. It has ADAS, 7 airbags and ventilated seats.
      2. Audi Q5 Premium Plus — ₹65.55 lakh — 36% below fair price
          ₹65.55 lakh is 36% below its fair price of ₹1.03 crore, so you save about ₹37.65 lakh on what these features
          usually cost. It has a 5-star Global NCAP rating, 8 airbags and a panoramic sunroof.
      3. Audi Q

### Above ₹1 crore

In [33]:
show_segment(7)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Above ₹1 crore: 92 variants of 29 models from 9 brands, ₹1.00 crore to ₹4.90 crore
Average price ₹2.09 crore, standard deviation ₹95.40 lakh
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▌ VALUE FOR MONEY

  S01  10 most value for money variants, in order
      1. BMW X7 xDrive40i M Sport Signature — ₹1.26 crore — 41% below fair price
          ₹1.26 crore is 41% below its fair price of ₹2.13 crore, so you save about ₹87.57 lakh on what these features
          usually cost. It has a 5-star Global NCAP rating, 9 airbags and a sunroof.
      2. Mercedes-Benz GLS 450 4MATIC AMG Line — ₹1.38 crore — 37% below fair price
          ₹1.38 crore is 37% below its fair price of ₹2.21 crore, so you save about ₹83.34 lakh on what these features
          usually cost. It has ADAS, 10 airbags and a panoramic

## Part 7: Brand insights

The same 26 questions for each brand, with "segment" read as "brand". Comparisons are made within the brand's own range: a "weak engine" here means weaker than most of that brand's SUVs, and "the most expensive variants" means the brand's priciest quarter. Small brands get the small-group rule: when fewer cars qualify than a list asks for, only the best one is shown.

In [34]:
brands = describe_groups(cars, "company").rename(columns={"company": "brand"})
brand_segments = cars.groupby("company")["segment_id"].agg(lambda s: sorted(s.unique()))
brands["segments"] = brands["brand"].map(lambda b: "; ".join(SEGMENTS.set_index("segment_id").loc[brand_segments[b], "segment"]))
brands = brands.sort_values("brand").reset_index(drop=True)
print(brands.assign(from_lakh=brands["from_price"] / 1e5, to_lakh=brands["to_price"] / 1e5)
      [["brand", "variants", "models", "from_lakh", "to_lakh", "segments"]].round(2).to_string(index=False))

brand_groups = [
    Group("brand", b, cars[cars["company"] == b], among="from this brand", where="", pool="this brand's SUVs")
    for b in brands["brand"]
]
brand_answers = pd.DataFrame([row for G in brand_groups for row in answer_group(G)], columns=ANSWER_COLUMNS)
print(f"\n{len(brand_answers):,} answer rows for {len(brand_groups)} brands.")

        brand  variants  models  from_lakh  to_lakh                                                                                             segments
         Audi        10       7      51.99   233.83                                                                     ₹50 lakh–1 crore; Above ₹1 crore
          BMW        17       9      49.90   254.55                                                        ₹30–50 lakh; ₹50 lakh–1 crore; Above ₹1 crore
          BYD         7       2      24.99    55.90                                                           ₹20–30 lakh; ₹30–50 lakh; ₹50 lakh–1 crore
      Citroen        30       3       8.55    37.32                                                             Under ₹10 lakh; ₹10–15 lakh; ₹30–50 lakh
        Honda        12       2      11.80    50.00                                                                ₹10–15 lakh; ₹15–20 lakh; ₹30–50 lakh
      Hyundai       185       8       5.81    55.70                              U


2,388 answer rows for 25 brands.


In [35]:
for b in brands.itertuples():
    print("\n" + "═" * 118)
    prices = lakh(b.from_price) if b.from_price == b.to_price else f"{lakh(b.from_price)} to {lakh(b.to_price)}"
    print(f"{b.brand}: {plural(b.variants, 'variant')} of {plural(b.models, 'model')}, {prices}")
    print("═" * 118)
    show(brand_answers, "brand", b.brand)


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Audi: 10 variants of 7 models, ₹51.99 lakh to ₹2.34 crore
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▌ VALUE FOR MONEY

  B01  10 most value for money variants, in order
      1. Audi Q5 Premium Plus — ₹65.55 lakh — 36% below fair price
          ₹65.55 lakh is 36% below its fair price of ₹1.03 crore, so you save about ₹37.65 lakh on what these features
          usually cost. It has a 5-star Global NCAP rating, 8 airbags and a panoramic sunroof.
      2. Audi Q5 Technology — ₹70.73 lakh — 33% below fair price
          ₹70.73 lakh is 33% below its fair price of ₹1.06 crore, so you save about ₹35.17 lakh on what these features
          usually cost. It has 8 airbags, a panoramic sunroof and a torque-converter automatic.
      3. Audi Q7 Premium Plus — ₹87.17 lakh — 24% below fair price
     

## Part 8: Overall insights

These questions look at all SUVs together, regardless of segment or brand.

In [36]:
overall_group = Group("overall", "All SUVs", cars, among="in the market", where="", pool="all SUVs")
overall_answers = pd.DataFrame(answer_group(overall_group), columns=ANSWER_COLUMNS)
print(f"All SUVs: {len(cars):,} variants of {cars['model_name'].nunique()} models from {cars['company'].nunique()} brands")
show(overall_answers, "overall", "All SUVs")

All SUVs: 1,399 variants of 136 models from 25 brands

▌ GEARBOX TYPE

  O01  At what price does the DCT automatic gearbox start? First 3 variants overall
      • DCT automatics start at ₹10.00 lakh in the market, with the Kia Sonet HTK (O) Turbo DCT.
      1. Kia Sonet HTK (O) Turbo DCT — ₹10.00 lakh — DCT
          7-speed DCT, petrol, 118 bhp.
      2. Hyundai Venue HX 5 Turbo DCT — ₹10.95 lakh — DCT
          7-speed DCT, petrol, 118 bhp.
      3. Kia Sonet HTK Plus (O) Turbo DCT — ₹11.09 lakh — DCT
          7-speed DCT, petrol, 118 bhp.

  O02  At what price does the CVT automatic gearbox start? First 3 variants overall
      • CVT automatics start at ₹9.30 lakh in the market, with the Nissan Magnite Acenta Turbo CVT. Strong hybrids use
        their own hybrid gearbox, so they aren't counted as CVTs.
      1. Nissan Magnite Acenta Turbo CVT — ₹9.30 lakh — CVT
          CVT, petrol, 99 bhp.
      2. Renault Kiger Techno Turbo CVT — ₹9.40 lakh — CVT
          CVT, petrol, 99 bhp.


## Part 9: Checks and saving the files

Before saving, a few checks make sure the answers are complete and consistent:
1. Every question has an answer, or a note saying why there is none, in every segment and brand.
2. Ranks run 1, 2, 3… without gaps, and no list is longer than it asks for.
3. Every answer row has a reason, and every named variant exists in `variants.csv`.
4. The small-group rule holds: a list is either full, or shows just one item and explains why.
5. A spot check: the top value-for-money variant in each segment, recomputed directly from the data.

In [37]:
answers = pd.concat([segment_answers, brand_answers, overall_answers], ignore_index=True)

# 1. every question answered (or explained) in every group
expected = {(G.tab, G.name, q) for G in segment_groups + brand_groups + [overall_group]
            for q in questions.loc[questions["tab"] == G.tab, "question_id"]}
present = set(map(tuple, answers[["tab", "group", "question_id"]].drop_duplicates().to_numpy()))
assert expected == present, f"missing answers: {sorted(expected - present)[:5]}"

# 2. ranks run 1..n and never exceed the list size
ranked = answers[answers["item_type"] != "note"]
sizes = questions.set_index(["tab", "question_id"])["list_size"]
for (tab, group, qid), block in ranked.groupby(["tab", "group", "question_id"]):
    assert list(block["rank"]) == list(range(1, len(block) + 1)), (tab, group, qid)
    n = sizes[(tab, qid)]
    assert n == 0 or len(block) <= n, (tab, group, qid, len(block), n)
    # 4. small-group rule: full list, or exactly one item with a note explaining it
    if n and len(block) < n:
        has_note = answers[(answers["tab"] == tab) & (answers["group"] == group) & (answers["question_id"] == qid) & (answers["item_type"] == "note")]
        assert len(block) == 1 and len(has_note), (tab, group, qid)

# 3. reasons present; variants exist
assert answers["reason"].fillna("").str.len().gt(10).all()
assert set(ranked.loc[ranked["item_type"] == "variant", "name"]) <= set(variants_out["name"])

# 5. spot check against a direct calculation
for s in SEGMENTS.itertuples():
    direct = cars[(cars["segment_id"] == s.segment_id) & cars["fair_price"].notna()].sort_values(["difference_pct", "ex_showroom_price", "name"])["name"].iloc[0]
    segment_name = s.segment
    shown = answers.query("tab == 'segment' and group == @segment_name and question_id == 'S01' and rank == 1")["name"].iloc[0]
    assert direct == shown, (s.segment, direct, shown)

print("All checks passed.")
print(answers.groupby("tab").agg(groups=("group", "nunique"), rows=("rank", "size")).to_string())

All checks passed.
         groups  rows
tab                  
brand        25  2388
overall       1    62
segment       7  1014


Now the files. The app reads these directly:

| File | One row per | Used for |
|---|---|---|
| `segments.csv` | segment | the home page: each segment's size, price range, average price and standard deviation |
| `segment_models.csv` | model in a segment | clicking a segment: its models and variant counts |
| `brands.csv` | brand | the Brands tab's list of brands |
| `variants.csv` | variant | every variant's segment, specs and scores |
| `questions.csv` | question | each tab's questions, categories and order |
| `answers_segments.csv`, `answers_brands.csv`, `answers_overall.csv` | answer item or note | each question's ranked answers and reasons |
| `feature_lists.csv` | feature | what each score counts, for the app's explanation page |

In [38]:
outputs = {
    "segments.csv": segments[["segment_id", "segment", "from_lakh", "to_lakh", "variants", "models", "brands",
                              "from_price", "to_price", "avg_price", "std_price", "cheapest_variant"]].replace({"to_lakh": {np.inf: np.nan}}),
    "segment_models.csv": segment_models,
    "brands.csv": brands,
    "variants.csv": variants_out,
    "questions.csv": questions[["tab", "question_id", "order", "category", "category_order", "number", "question",
                                "asked_as", "kind", "list_size"]],
    "answers_segments.csv": segment_answers,
    "answers_brands.csv": brand_answers,
    "answers_overall.csv": overall_answers,
    "feature_lists.csv": feature_lists,
}
# Prices are whole rupees: save them without a trailing ".0".
for frame in outputs.values():
    for column in ["price", "ex_showroom_price", "on_road_price", "fair_price", "from_price", "to_price", "avg_price", "std_price"]:
        if column in frame.columns:
            frame[column] = frame[column].round().astype("Int64")
for file_name, frame in outputs.items():
    frame.to_csv(OUT_DIR / file_name, index=False)

# Read every file back the way the app will, and check nothing changed on the way.
for file_name, frame in outputs.items():
    back = pd.read_csv(OUT_DIR / file_name, **READ, dtype=AS_TEXT)
    assert back.shape == frame.shape, file_name
    print(f"Saved {file_name:<22} {back.shape[0]:>6,} rows x {back.shape[1]} columns")

readme = ["# insights_data", "",
          "Files written by `notebooks/suv_insights.ipynb`; the web app reads them. Do not edit them by hand: "
          "re-run the notebook (Restart & Run All) to rebuild them.", "",
          "- Source: CarDekho data cleaned by `notebooks/suv_research.ipynb` (`app_data/`), with its fair prices.",
          "- Prices are ex-showroom, in rupees. 1 lakh = ₹1,00,000; 1 crore = ₹1,00,00,000.",
          "- In the answers files, rows with `item_type` = note (rank 0) sum up an answer or explain a short or empty list.",
          "- In `segments.csv`, an empty `to_lakh` means the segment has no upper price limit.", ""]
for file_name, frame in outputs.items():
    readme += [f"## {file_name}", "", f"Rows: {len(frame):,}", "", "Columns: " + ", ".join(f"`{c}`" for c in frame.columns), ""]
(OUT_DIR / "README.md").write_text("\n".join(readme), encoding="utf-8")
print("Saved README.md")

Saved segments.csv                7 rows x 12 columns
Saved segment_models.csv        213 rows x 8 columns
Saved brands.csv                 25 rows x 8 columns
Saved variants.csv            1,399 rows x 41 columns
Saved questions.csv              60 rows x 10 columns
Saved answers_segments.csv    1,014 rows x 13 columns
Saved answers_brands.csv      2,388 rows x 13 columns
Saved answers_overall.csv        62 rows x 13 columns
Saved feature_lists.csv          64 rows x 3 columns
Saved README.md
